# KYC identity extraction — production pipeline

Single self-contained notebook. Offline, Domino Data Lab, local Qwen VL checkpoint, H100.

**Governing principle, enforced in code and not only in the prompt:**

> accuracy > hallucination avoidance > completeness > speed
>
> `UNREADABLE` and `LOW_CONFIDENCE` are correct answers. An invented identity value is not.
> Every failure path returns `null` plus a review flag. Nothing is ever auto-corrected.

| Section | Contents |
|---|---|
| 0 | Analysis: root causes, where the 6 minutes go, recommended architecture |
| A | Document intake: unzip, presence/absence report, target customers |
| 1 | `analyze_page_quality()` — pure CV, ~30 ms, no GPU |
| 2 | `detect_orientation()` + adaptive `preprocess_page()` |
| 3 | Strict extraction prompt + Qwen engine (loaded once, batched, logprobs, brace stop) |
| 4 | Parsing, coercion, field validators |
| 5 | Measured confidence scoring |
| 6 | Targeted retry ladder |
| 7 | Page-level orchestration and document aggregation |
| 8 | Logging, statistics, benchmark, calibration |

Run top to bottom. Set `CFG.mock_model = True` to rehearse the whole pipeline without a GPU.

## 0.1 — What is actually going wrong (Part 1)

### Root causes by symptom

| Symptom | Most likely cause | Layer that must fix it |
|---|---|---|
| Page unreadable | Effective text height falls below ~10 px: a 300 dpi A4 render is 2480×3509 and gets downscaled to 1600 px before the model sees it, losing 35% of linear resolution | image preprocessing (resolution policy, crop-then-scale) |
| Wrong values on poor scans | One globally enhanced, downscaled read, with no fallback ladder — a bad read is final | retry logic + confidence gating |
| Model "seems uncertain" | It is asked to self-report `high/medium/low`. That label is a generated token like any other; it correlates with fluency, not with legibility | confidence scoring — it must be computed, not requested |
| Hallucinated / completed characters | Free-form generation with nothing to check the answer against. A model that outputs only a value cannot be contradicted; a model that must also reproduce the glyphs it sees can be | prompting + post-processing validation |
| Repetition, looping, over-long output | `max_new_tokens=1024`, no stop condition, and — critically — **Qwen3 chat templates enable thinking by default**, so a `<think>` block can burn 400–900 tokens before the JSON starts | inference parameters |
| ~6 min/document | Output token count × number of model calls. Budget below | inference parameters, call-count reduction, batching |

### Which layer owns which problem

| Problem | PDF | Image | Prompt | Inference | Post | Validation | Retry |
|---|:-:|:-:|:-:|:-:|:-:|:-:|:-:|
| Rotation / upside down | | ✔ | | | | | |
| Skew | | ✔ | | | | | |
| Low resolution | ✔ | ✔ | | | | | ✔ |
| Blur / noise / contrast | | ✔ | | | | | ✔ |
| Guessed characters | | | ✔ | ✔ | ✔ | ✔ | |
| Fabricated evidence | | | ✔ | | ✔ | ✔ | |
| Ungrounded confidence | | | | ✔ (logprobs) | ✔ | ✔ | |
| Repetition / looping | | | ✔ | ✔ | | | |
| Latency | ✔ | ✔ | ✔ | ✔ | | | ✔ |
| Bad data reaching KYC | | | | | | ✔ | ✔ |

Note what is **not** in this table: a separate OCR engine. Tesseract is used here only for page
orientation, never for transcription — running a classical OCR pass beside the VLM would add a
second source of unverifiable text, and merging two disagreeing transcriptions is precisely the
situation where systems start inventing values.

## 0.2 — Where the six minutes go, and the plan (Parts 11, 12)

Budget for a typical 5-page identity document on one H100, bf16, HF `transformers`, batch 1.
Decode throughput for a 27B-class model is realistically **20–35 tok/s** — weights ≈54 GB, so
decoding is HBM-bandwidth bound, plus per-token Python overhead (no CUDA graphs by default).

| Item | Per page | ×5 pages | Share |
|---|---|---|---|
| PDF render at 300 dpi | 0.4 s | 2 s | 1% |
| CLAHE + bilateral filter **at full 8.7 MP**, before downscaling | 1.5–3 s | 10 s | 3% |
| VLM orientation probe — full image prefill for 4 output tokens | 1.5 s | 8 s | 2% |
| Vision encoder + prefill (≈2 300 visual + ≈1 400 text tokens) | 0.8 s | 4 s | 1% |
| **Decode**: 1 024 max tokens, thinking block, no stop condition | **25–40 s** | **150–200 s** | **~60%** |
| Full-page retry on weak pages (≈40% of pages) | +30 s | 60 s | 20% |
| Merge, I/O | 0.3 s | 2 s | 1% |
| **Total** | | **≈ 240–290 s** | |

One slow page — blur, empty result, retry, still empty — takes that to six minutes.

**The dominant term is output tokens, not image size.** Roughly 80% of wall-clock is
autoregressive decoding. Every optimisation below is ranked against that fact.

### Optimisation plan, ordered by expected impact

Percentages are **targets to verify with `run_benchmark()` in Stage 8**, not measurements.

| # | Change | Mechanism | Expected |
|---|---|---|---|
| 1 | Thinking off, `max_new_tokens` 384, brace stop, `{` prefill | cuts decoded tokens per call 2–4× | largest single win |
| 2 | CV orientation cascade | removes one model call per page | −15–25% |
| 3 | Targeted retry: failed pages only, missing fields only, 160 tokens | retries stop costing a full page | −10–20% on retrying docs |
| 4 | Blank-page triage before any GPU work | zero calls for empty pages | corpus dependent |
| 5 | Preprocess *after* downscaling | CLAHE + bilateral on 1.2 MP not 8.7 MP | −8–12 s/doc of CPU |
| 6 | `max_pixels` cap + document crop | shorter prefill, better effective resolution | −5–10% |
| 7 | `page_batch_size=4` | batch-1 decode wastes H100 bandwidth; batching amortises weight reads | −20–40% multi-page |
| 8 | *Optional* vLLM backend, same local weights | continuous batching, paged KV, CUDA graphs, prefix caching | often 2–4× beyond the above |

### Resolution and visual tokens (Part 12)

For Qwen-VL one visual token ≈ a 28×28 pixel block (14×14 patches, 2×2 merged), so
`visual_tokens ≈ W × H / 784`:

| Render | Pixels | Visual tokens | Small-print legibility |
|---|---|---|---|
| 300 dpi A4, untouched | 2480×3509 | ≈ 11 100 | best, unaffordable |
| 1600 px max side | 1600×1130 | ≈ 2 300 | good |
| **1280 px max side (default)** | 1280×905 | ≈ 1 480 | good printed, marginal MRZ |
| 1024 px max side | 1024×724 | ≈ 950 | risky |

The default is `target_max_dimension = 1280` **plus a document-region crop applied first**.
Cropping is the free lunch: an ID card filling 40% of an A4 scan gains ~1.6× effective resolution
at a *lower* token count than the whole page. Do not take 1 280 on faith —
`sweep_resolution()` in Stage 8 runs your labelled sample at 1024/1280/1600/2000 and prints
accuracy against latency. Pick the smallest dimension whose accuracy matches the largest.

### Batching and memory on the H100

A 27B bf16 model is ≈54 GB of weights; on an 80 GB card that leaves ~20–25 GB for activations and
KV cache. At ~1 500 visual tokens per page, batch 4 is comfortable and batch 8 is where to start
watching `nvidia-smi`. The code falls back to sequential on OOM automatically. Greedy decoding
with left padding should be numerically stable across batch sizes — but benchmark
`page_batch_size` 1 against 4 and confirm rather than assume.

The model is loaded **once per process** (`QwenVLEngine.get()` caches it). Never call
`load_model()` inside a loop.

## 0.3 — Recommended architecture

```
                    ┌──────────────────────────────────────────┐
                    │  Model loaded ONCE (process singleton)    │
                    └──────────────────────────────────────────┘
                                      │
PDF ──► render page ──► analyze_page_quality()   ← pure CV, ~30 ms, no GPU
                              │
                              ├──► blank / no ink? ──► NOT_PRESENT, 0 model calls
                              │
                              ├──► detect_orientation()  ← CV first, VLM only to break a tie
                              │
                              └──► plan_preprocessing()  ← only what the metrics justify
                                          │
                                    preprocess_page()
                                          │
                    ┌─────────────────────┴──────────────────────┐
                    │  extract_page() — strict JSON, evidence,    │
                    │  5 statuses, greedy decode, brace stop      │
                    └─────────────────────┬──────────────────────┘
                                          │
                              parse + coerce (fail closed)
                                          │
                              validate_extraction()   ← format, cross-field, evidence
                                          │
                              score_confidence()      ← 5 measured components
                                          │
                              should_retry()? ── no ──► accept page
                                          │ yes
                              retry_page()  ← ladder, ONLY missing fields, ONLY this page
                                          │
                              aggregate_document_results()
                                          │
                        PASS / LOW_CONFIDENCE / UNREADABLE / MANUAL_REVIEW
```

Two invariants hold everywhere in this notebook:

1. **Fail closed.** Unparsable JSON, inference error, page conflict, evidence mismatch — all
   produce `null` plus a review flag, never a partial guess.
2. **Nothing is corrected.** Validators and checksums *flag only*. No `O`→`0`, no date
   reformatting, no MRZ repair, no transliteration. The only thing that could justify changing a
   character is the pixels, and a validator cannot see them.

## Stage 0 — Configuration, timing, call accounting

**Problem.** "Why does a document take 6 minutes?" was unanswerable because nothing was measured
per stage. Optimising without a breakdown is guesswork.

**Solution.** A single `Config` dataclass holds every tunable (no constants buried in functions),
and every expensive operation runs inside a `Timer` that writes into a per-document trace. Model
calls and decoded tokens are counted centrally, so `model_inference_time`, `n_model_calls` and
`n_output_tokens` are facts rather than estimates.

In [ ]:
# =========================================================================
# STAGE 0.1 — CONFIGURATION
# =========================================================================
from __future__ import annotations

import os
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

# Must be set BEFORE torch is imported, so it lives in the very first cell.
# expandable_segments lets the caching allocator grow segments instead of fragmenting, which is
# what turns "plenty free but still OOM" into a working run on long, variable-length prompts.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple


@dataclass
class Config:
    # ---------------- paths ------------------------------------------------
    zip_path: Path  = Path("/domino/datasets/local/kyc/kyc_documents.zip")
    out_dir: Path   = Path("/domino/datasets/local/kyc/run")
    selected_dir: Optional[Path] = None      # defaults to out_dir/00_selected_documents
    model_path: str = "/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen3.8-27B/main"
    doc_name: str   = "JUSTIFICATIF IDENTITE.PDF"
    doc_key: str    = "JUSTIFICATIF_IDENTITE"

    # ---------------- document intake (stage A) ---------------------------
    enable_fuzzy_filename_match: bool = True
    fuzzy_threshold: float = 0.88            # fuzzy hits are ALWAYS flagged for review
    expand_nested_zips: bool = True

    # ---------------- rendering & resolution ------------------------------
    render_dpi: int = 300                 # render high, then crop+downscale: never the reverse
    target_max_dimension: int = 1280      # ~1480 visual tokens; see the resolution table in 0.2
    retry_max_dimension: int = 1600       # attempt 3 of the ladder
    min_text_height_px: float = 11.0      # below this, upscale rather than send unreadable glyphs
    max_upscale_factor: float = 2.0       # Lanczos only; no super-resolution (it invents glyphs)
    max_visual_tokens: int = 1280         # hard cap passed to the processor as max_pixels
    max_pages: int = 12

    # ---------------- quality thresholds (calibrate on your corpus) -------
    # All measured on a fixed 1000px working copy so thresholds stay comparable across scans.
    blur_var_poor: float = 80.0           # variance of Laplacian below this = blurred
    blur_var_good: float = 300.0
    contrast_low: float = 0.30            # (p95-p5)/255
    contrast_excessive: float = 0.95
    noise_sigma_high: float = 6.0         # Immerkaer estimate
    illum_uniformity_poor: float = 0.14   # std of 16x16 block means / 255
    dark_background_ratio: float = 0.55
    ink_ratio_blank: float = 0.002        # combined with "no text components": skip, 0 GPU calls
    deskew_min_deg: float = 0.4
    deskew_max_deg: float = 15.0
    orientation_margin_min: float = 0.08  # below this the CV 180-degree test is a coin flip
    osd_conf_min: float = 2.0

    # ---------------- inference (rationale in the Stage 3 table) ----------
    torch_dtype: str = "bfloat16"
    device_map: str = "auto"
    attn_impl: Optional[str] = "sdpa"     # "flash_attention_2" if installed
    max_new_tokens: int = 384             # sized to the schema, not to a round number
    max_new_tokens_targeted: int = 160    # retries ask for fewer fields -> fewer tokens
    max_new_tokens_probe: int = 4         # orientation tie-break only
    repetition_penalty: float = 1.0       # MUST be 1.0: >1 corrupts 1980, <<<, repeated digits
    no_repeat_ngram_size: int = 0         # MUST be 0: would forbid << in an MRZ
    disable_thinking: bool = True
    json_prefill: str = "{"               # forces the answer to open as JSON
    # Batching is an optimisation to switch ON after measuring headroom with
    # autotune_batch_size(), not a default. Batch 1 is the safe starting point: every page in a
    # batch holds its own vision activations and KV cache simultaneously.
    page_batch_size: int = 1              # 0/1 = sequential. Splits automatically on OOM.
    mock_model: bool = False

    # ---------------- GPU memory ------------------------------------------
    # Reserve VRAM that device_map="auto" is NOT allowed to fill with weights. Without this,
    # accelerate packs the card with layers and leaves nothing for the vision encoder, the KV
    # cache and the logits tensor -- the classic "weights load fine, every generate() OOMs".
    reserve_vram_gib: float = 8.0
    cpu_offload_gib: float = 64.0         # overflow budget when weights do not fit the reserve
    oom_downscale_factor: float = 0.65    # emergency image shrink after an OOM
    oom_max_downscales: int = 2
    max_consecutive_error_docs: int = 2   # circuit breaker: stop the batch, do not burn the queue

    # ---------------- retry ladder ----------------------------------------
    max_attempts_per_page: int = 3        # attempt 1 + up to 2 escalations
    enable_second_pass: bool = True       # independent verification read when still uncertain
    retry_only_missing_fields: bool = True
    skip_retry_if_hopeless: bool = True   # blank / near-black pages never enter the ladder

    # ---------------- confidence (rationale in the Stage 5 table) ---------
    weights: Dict[str, float] = field(default_factory=lambda: {
        "visual_clarity":   0.20,
        "evidence_support": 0.30,
        "token_confidence": 0.20,
        "format_validity":  0.20,
        "consistency":      0.10,
    })
    band_high: float = 0.80
    band_medium: float = 0.60
    band_low: float = 0.40
    cap_uncertain: float = 0.55
    cap_evidence_gap: float = 0.45        # evidence contains '?'
    cap_format_invalid: float = 0.50
    cap_no_evidence: float = 0.40
    cap_handwritten: float = 0.60
    accept_band: str = "MEDIUM"           # a field below this is not accepted into final_fields

    # ---------------- review policy ---------------------------------------
    critical_fields: Tuple[str, ...] = ("surname", "given_names", "date_of_birth",
                                        "document_number")
    max_unreadable_critical: int = 0      # any unreadable critical field -> manual review

    def __post_init__(self) -> None:
        self.zip_path = Path(self.zip_path)
        self.out_dir = Path(self.out_dir)
        self.selected_dir = Path(self.selected_dir) if self.selected_dir \
            else self.out_dir / "00_selected_documents"
        self.selected_dir.mkdir(parents=True, exist_ok=True)

    def dirs(self) -> Dict[str, Path]:
        d = {k: self.out_dir / v for k, v in {
            "pages": "pages", "results": "results", "reports": "reports",
            "logs": "logs", "bench": "benchmark"}.items()}
        for p in d.values():
            p.mkdir(parents=True, exist_ok=True)
        return d


CFG = Config()
DIRS = CFG.dirs()
print("zip_path     :", CFG.zip_path, "| exists:", CFG.zip_path.exists())
print("out_dir      :", CFG.out_dir)
print("selected_dir :", CFG.selected_dir)
print("model_path   :", CFG.model_path, "| exists:", Path(CFG.model_path).exists())

In [ ]:
# =========================================================================
# STAGE 0.2 — IMPORTS, TIMING, CALL ACCOUNTING
# =========================================================================
import io, re, json, math, time, logging, hashlib, difflib, unicodedata, traceback
from collections import OrderedDict, defaultdict
from contextlib import contextmanager
from datetime import datetime, timezone, date

import numpy as np
import pandas as pd
from PIL import Image

Image.MAX_IMAGE_PIXELS = None

try:
    import pymupdf as fitz
except Exception:
    try:
        import fitz
    except Exception:
        fitz = None
try:
    import cv2
except Exception:
    cv2 = None
try:
    import pytesseract
    from pytesseract import Output as TessOutput
    pytesseract.get_tesseract_version()
except Exception:
    pytesseract, TessOutput = None, None
try:
    import torch
except Exception:
    torch = None

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
utcnow = lambda: datetime.now(timezone.utc).isoformat()

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)-7s %(message)s",
                    datefmt="%H:%M:%S")
log = logging.getLogger("kyc")


class Trace:
    """Per-document timing and call accounting. Every number in the run log comes from here."""

    def __init__(self, customer_id: str, document: str):
        self.customer_id = customer_id
        self.document = document
        self.t0 = time.perf_counter()
        self.timings: Dict[str, float] = defaultdict(float)
        self.counts: Dict[str, int] = defaultdict(int)
        self.page_times: Dict[str, float] = {}
        self.events: List[Dict[str, Any]] = []

    @contextmanager
    def timed(self, key: str):
        t = time.perf_counter()
        try:
            yield
        finally:
            self.timings[key] += time.perf_counter() - t

    def count(self, key: str, n: int = 1) -> None:
        self.counts[key] += n

    def event(self, kind: str, **kw) -> None:
        self.events.append({"t": round(time.perf_counter() - self.t0, 3), "kind": kind, **kw})

    def summary(self) -> Dict[str, Any]:
        return {
            "customer_id": self.customer_id,
            "document": self.document,
            "processing_time_s": round(time.perf_counter() - self.t0, 2),
            "timings_s": {k: round(v, 3) for k, v in sorted(self.timings.items())},
            "page_processing_times_s": {k: round(v, 2) for k, v in self.page_times.items()},
            "counts": dict(self.counts),
        }


def strip_accents(text: str) -> str:
    return "".join(c for c in unicodedata.normalize("NFKD", str(text))
                   if not unicodedata.combining(c))


def compare_key(value: Any) -> str:
    """Comparison key only. NEVER stored, never returned as a value."""
    return re.sub(r"[^A-Z0-9<]+", "", strip_accents(str(value)).upper())


print(f"run_id={RUN_ID} | cv2={'ok' if cv2 else 'MISSING'} | "
      f"pymupdf={'ok' if fitz else 'MISSING'} | tesseract={'ok' if pytesseract else 'absent'} | "
      f"torch={torch.__version__ if torch else 'MISSING'} | "
      f"cuda={torch.cuda.is_available() if torch else False}")

## Stage A — Document intake

Unchanged in behaviour from the working v1 implementation (Part 19: do not replace what works),
restated here so the notebook is self-contained.

Scanned KYC folders are messy: accents, underscores, mixed case, trailing `(1)`, occasional
typos. Filenames are normalised — accent-stripped, upper-cased, punctuation collapsed — and
matched against a controlled alias table at three levels, all three recorded so nothing slips
through silently:

* `exact` — normalised name equals a known alias
* `contains` — an alias appears inside the name (`JUSTIFICATIF IDENTITE 001`)
* `fuzzy` — `difflib` ratio ≥ `fuzzy_threshold`, **always flagged for human review**

The source specification spells one file `CARTON SIGNATUTE.PDF` (a typo for *SIGNATURE*); both
spellings are aliases so folders match either way. Only `JUSTIFICATIF IDENTITE.PDF` proceeds to
extraction — the other four are recorded present/absent and left untouched.

In [ ]:
# =========================================================================
# STAGE A.1 — DOCUMENT CATALOG AND FILENAME MATCHING
# =========================================================================
ALLOWED_EXT = {".pdf"}


def norm_name(text: str) -> str:
    """Accent-free, upper-case, punctuation-collapsed form. Used ONLY to match filenames."""
    s = strip_accents(text).upper()
    s = re.sub(r"[^A-Z0-9]+", " ", s)
    return re.sub(r"\s+", " ", s).strip()


REQUIRED_DOCS: "OrderedDict[str, Dict[str, Any]]" = OrderedDict([
    ("JUSTIFICATIF_IDENTITE", {
        "label": "JUSTIFICATIF IDENTITE.PDF",
        "aliases": ["JUSTIFICATIF IDENTITE", "JUSTIFICATIF D IDENTITE", "JUSTIFICATIF DE IDENTITE",
                    "JUSTIF IDENTITE", "PIECE IDENTITE", "PIECE D IDENTITE", "IDENTITE"]}),
    ("JUSTIFICATIF_DOMICILE", {
        "label": "JUSTIFICATIF DOMICILE.PDF",
        "aliases": ["JUSTIFICATIF DOMICILE", "JUSTIFICATIF DE DOMICILE", "JUSTIF DOMICILE",
                    "PREUVE DE DOMICILE", "DOMICILE"]}),
    ("CONVENTION_COMPTE", {
        "label": "CONVENTION COMPTE.PDF",
        "aliases": ["CONVENTION COMPTE", "CONVENTION DE COMPTE", "CONVENTION DU COMPTE",
                    "CONVENTION OUVERTURE COMPTE"]}),
    ("FATCA", {
        "label": "FATCA.PDF",
        "aliases": ["FATCA", "FORMULAIRE FATCA", "FATCA CRS", "AUTOCERTIFICATION FATCA"]}),
    ("CARTON_SIGNATURE", {
        "label": "CARTON SIGNATUTE.PDF",          # typo preserved from the source specification
        "aliases": ["CARTON SIGNATUTE", "CARTON SIGNATURE", "CARTON DE SIGNATURE",
                    "SPECIMEN SIGNATURE", "SPECIMEN DE SIGNATURE"]}),
])
for _k, _spec in REQUIRED_DOCS.items():
    _spec["norm_aliases"] = sorted({norm_name(a) for a in [_spec["label"]] + _spec["aliases"]},
                                   key=len, reverse=True)
DOC_KEYS: List[str] = list(REQUIRED_DOCS)


def match_required_doc(filename: str, cfg: Config = CFG) -> Tuple[Optional[str], str, float]:
    """Return (doc_key, match_type, score). doc_key is None when the file is not required."""
    p = Path(str(filename))
    if p.suffix.lower() not in ALLOWED_EXT:
        return None, "wrong_extension", 0.0
    stem = re.sub(r"\s+\d+$", "", norm_name(p.stem)).strip()   # drop trailing copy counters

    for key, spec in REQUIRED_DOCS.items():
        if stem in spec["norm_aliases"]:
            return key, "exact", 1.0
    for key, spec in REQUIRED_DOCS.items():
        for alias in spec["norm_aliases"]:
            if len(alias) >= 5 and alias in stem:
                return key, "contains", 0.95
    if cfg.enable_fuzzy_filename_match:
        best_key, best = None, 0.0
        for key, spec in REQUIRED_DOCS.items():
            for alias in spec["norm_aliases"]:
                r = difflib.SequenceMatcher(None, stem, alias).ratio()
                if r > best:
                    best_key, best = key, r
        if best >= cfg.fuzzy_threshold:
            return best_key, "fuzzy", round(best, 4)
    return None, "no_match", 0.0


for _t in ["JUSTIFICATIF IDENTITE.PDF", "justificatif_identité (1).pdf", "CARTON SIGNATUTE.PDF",
           "Carton-Signature.PDF", "convention de compte v2.pdf", "RIB.pdf", "photo.jpg"]:
    print(f"{_t:<36} -> {match_required_doc(_t)}")

In [ ]:
# =========================================================================
# STAGE A.2 — SAFE ZIP EXTRACTION (only the required documents are written to disk)
# =========================================================================
import zipfile


def _safe_relpath(member_name: str) -> Optional[List[str]]:
    """Reject absolute paths, drive letters and .. traversal (zip-slip)."""
    name = member_name.replace("\\", "/")
    if name.startswith("/") or re.match(r"^[A-Za-z]:", name):
        return None
    parts = [p for p in name.split("/") if p not in ("", ".")]
    return None if any(p == ".." for p in parts) else parts


def _detect_root_prefix(all_parts: List[List[str]]) -> Optional[str]:
    """If every member sits under a single wrapper folder, strip it."""
    roots = {p[0] for p in all_parts if len(p) > 1}
    if len(roots) == 1:
        root = roots.pop()
        if any(len(p) > 2 and p[0] == root for p in all_parts):
            return root
    return None


def _iter_zip_entries(zip_path: Path, cfg: Config = CFG):
    """Yield (path_parts, filename, size, reader). Handles one level of nested ZIPs."""
    with zipfile.ZipFile(zip_path) as zf:
        infos, parts_list = [], []
        for info in zf.infolist():
            if info.is_dir():
                continue
            parts = _safe_relpath(info.filename)
            if parts is None:
                log.warning("unsafe zip member skipped: %s", info.filename)
                continue
            infos.append(info)
            parts_list.append(parts)
        root = _detect_root_prefix(parts_list)
        for info, parts in zip(infos, parts_list):
            if root and parts and parts[0] == root:
                parts = parts[1:]
            if not parts:
                continue
            if cfg.expand_nested_zips and parts[-1].lower().endswith(".zip"):
                try:
                    with zipfile.ZipFile(io.BytesIO(zf.read(info))) as izf:
                        for iinfo in izf.infolist():
                            if iinfo.is_dir():
                                continue
                            iparts = _safe_relpath(iinfo.filename)
                            if iparts is None:
                                continue
                            combined = parts[:-1] + iparts
                            payload = izf.read(iinfo)
                            yield combined, combined[-1], iinfo.file_size, (lambda b=payload: b)
                    continue
                except Exception as exc:
                    log.warning("nested zip unreadable %s: %s", "/".join(parts), exc)
            yield parts, parts[-1], info.file_size, (lambda i=info: zf.read(i))


def extract_required_documents(zip_path: Path = None,
                               cfg: Config = CFG) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Walk the archive, copy ONLY the five required PDFs, catalogue everything seen."""
    zip_path = Path(zip_path or cfg.zip_path)
    assert zip_path.exists(), f"ZIP not found: {zip_path}"
    all_rows, sel_rows, seen = [], [], {}

    for parts, fname, size, read_fn in _iter_zip_entries(zip_path, cfg):
        customer_id = parts[0] if len(parts) > 1 else "_ARCHIVE_ROOT_"
        doc_key, match_type, score = match_required_doc(fname, cfg)
        all_rows.append(dict(customer_id=customer_id, member="/".join(parts), filename=fname,
                             ext=Path(fname).suffix.lower(), size_bytes=int(size),
                             matched_doc_key=doc_key or "", match_type=match_type,
                             match_score=score))
        if doc_key is None:
            continue
        dest_dir = cfg.selected_dir / customer_id
        dest_dir.mkdir(parents=True, exist_ok=True)
        n_prev = seen.get((customer_id, doc_key), 0)
        seen[(customer_id, doc_key)] = n_prev + 1
        dest = dest_dir / (f"{doc_key}.pdf" if n_prev == 0 else f"{doc_key}__dup{n_prev+1}.pdf")
        data = read_fn()
        dest.write_bytes(data)
        sel_rows.append(dict(customer_id=customer_id, doc_key=doc_key,
                             source_member="/".join(parts), source_filename=fname,
                             extracted_path=str(dest), match_type=match_type, match_score=score,
                             size_bytes=len(data), is_duplicate=bool(n_prev),
                             sha256=hashlib.sha256(data).hexdigest()))
    return pd.DataFrame(sel_rows), pd.DataFrame(all_rows)

In [ ]:
# =========================================================================
# STAGE A.3 — PRESENCE / ABSENCE REPORT
# =========================================================================
def build_inventory(all_members_df: pd.DataFrame,
                    selected_df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    customers = sorted(set(all_members_df.get("customer_id", pd.Series(dtype=str))) |
                       set(selected_df.get("customer_id", pd.Series(dtype=str))))
    by_cd = {}
    if not selected_df.empty:
        for (cid, dk), grp in selected_df.groupby(["customer_id", "doc_key"]):
            by_cd[(cid, dk)] = grp

    rows, long_rows = [], []
    for cid in customers:
        n_files = int((all_members_df["customer_id"] == cid).sum()) if not all_members_df.empty else 0
        row: Dict[str, Any] = OrderedDict(customer_id=cid, n_files_in_folder=n_files)
        n_present, fuzzy = 0, []
        for dk in DOC_KEYS:
            grp = by_cd.get((cid, dk))
            present = grp is not None and len(grp) > 0
            row[dk] = "PRESENT" if present else "MISSING"
            row[f"{dk}__n_copies"] = int(len(grp)) if present else 0
            row[f"{dk}__source_filename"] = grp.iloc[0]["source_filename"] if present else ""
            row[f"{dk}__match_type"] = grp.iloc[0]["match_type"] if present else ""
            if present:
                n_present += 1
                if (grp["match_type"] == "fuzzy").any():
                    fuzzy.append(dk)
            long_rows.append(dict(customer_id=cid, document=REQUIRED_DOCS[dk]["label"], doc_key=dk,
                                  status="PRESENT" if present else "MISSING",
                                  n_copies=int(len(grp)) if present else 0,
                                  source_filename=grp.iloc[0]["source_filename"] if present else "",
                                  match_type=grp.iloc[0]["match_type"] if present else ""))
        row["n_required_present"] = n_present
        row["n_required_missing"] = len(DOC_KEYS) - n_present
        row["folder_complete"] = (n_present == len(DOC_KEYS))
        row["is_target_customer"] = (row[CFG.doc_key] == "PRESENT")
        row["fuzzy_matched_docs"] = ";".join(fuzzy)
        row["needs_filename_review"] = bool(fuzzy)
        rows.append(row)
    return pd.DataFrame(rows), pd.DataFrame(long_rows)


def run_intake(cfg: Config = CFG) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Unzip -> select -> report. Writes CSV/JSON/XLSX into the reports directory."""
    t0 = time.perf_counter()
    selected_df, all_members_df = extract_required_documents(cfg.zip_path, cfg)
    inventory_df, inventory_long_df = build_inventory(all_members_df, selected_df)

    rep = DIRS["reports"]
    inventory_df.to_csv(rep / "01_document_presence_report.csv", index=False, encoding="utf-8-sig")
    inventory_long_df.to_csv(rep / "01b_document_presence_long.csv", index=False, encoding="utf-8-sig")
    all_members_df.to_csv(rep / "00_archive_file_catalog.csv", index=False, encoding="utf-8-sig")
    (rep / "01_document_presence_report.json").write_text(
        inventory_df.to_json(orient="records", force_ascii=False, indent=2), encoding="utf-8")
    try:
        with pd.ExcelWriter(rep / "01_document_presence_report.xlsx") as xw:
            inventory_df.to_excel(xw, sheet_name="presence", index=False)
            inventory_long_df.to_excel(xw, sheet_name="detail", index=False)
    except Exception as exc:
        log.info("xlsx skipped: %s", exc)

    log.info("intake: %d members, %d customers, %d required PDFs kept, %d fuzzy matches (%.1fs)",
             len(all_members_df), inventory_df.shape[0], len(selected_df),
             int((selected_df["match_type"] == "fuzzy").sum()) if not selected_df.empty else 0,
             time.perf_counter() - t0)
    return inventory_df, selected_df


# inventory_df, selected_df = run_intake(CFG)
print("Stage A ready: call run_intake(CFG) to unzip and build the presence report")

In [ ]:
# =========================================================================
# STAGE A.4 — TARGET CUSTOMERS (those having JUSTIFICATIF IDENTITE.PDF)
# =========================================================================
def select_target_customers(inventory_df: pd.DataFrame, cfg: Config = CFG) -> pd.DataFrame:
    key = cfg.doc_key
    cols = ["customer_id", f"{key}__source_filename", f"{key}__match_type", f"{key}__n_copies"]
    tgt = inventory_df.loc[inventory_df[key] == "PRESENT", cols].copy()
    tgt.columns = ["customer_id", "source_filename", "match_type", "n_copies"]
    tgt["id_pdf_path"] = tgt["customer_id"].map(lambda c: str(cfg.selected_dir / c / f"{key}.pdf"))
    tgt = tgt.sort_values("customer_id").reset_index(drop=True)

    rep = DIRS["reports"]
    tgt.to_csv(rep / "02_target_customers.csv", index=False, encoding="utf-8-sig")
    (rep / "02_target_customers.json").write_text(
        json.dumps(tgt["customer_id"].tolist(), ensure_ascii=False, indent=2), encoding="utf-8")
    without = inventory_df.loc[inventory_df[key] != "PRESENT", "customer_id"].tolist()
    (rep / "02b_customers_without_identity_doc.json").write_text(
        json.dumps(without, ensure_ascii=False, indent=2), encoding="utf-8")
    log.info("targets: %d with identity document, %d without", len(tgt), len(without))
    return tgt


# target_df = select_target_customers(inventory_df, CFG)
print("Stage A.4 ready: call select_target_customers(inventory_df, CFG)")

## Stage 1 — Page quality assessment

**Problem.** v1 applied the same enhancement to every page. CLAHE on an already well-exposed scan
thins strokes; a bilateral filter on a clean 300 dpi render costs 1.5–3 s and gains nothing. Worse,
there was no way to know *why* a page failed — blur, resolution and contrast were never separated.

**Solution.** `analyze_page_quality()` returns the measurements that drive every downstream
decision. Three design points matter:

1. **Everything is measured on a fixed 1 000 px working copy.** Variance of Laplacian scales with
   image size, so thresholds are meaningless unless the measurement scale is fixed. This is the
   most common way blur thresholds silently break.
2. **`text_height_px` replaces DPI as the resolution signal.** A scan's stated DPI says nothing
   about legibility — a 600 dpi scan of a tiny card and a 200 dpi scan of an A4 form can carry the
   same glyph size. Median connected-component height is what actually predicts OCR failure.
3. **Blank-page detection is free money.** A page with `ink_ratio` under 0.4% carries no text; it
   is marked `NOT_PRESENT` and never reaches the GPU.

**Integration.** The returned dict feeds `plan_preprocessing()` (Stage 2), gates the retry ladder
(Stage 6) and supplies the `visual_clarity` component of the confidence score (Stage 5).

In [ ]:
# =========================================================================
# STAGE 1.1 — LOW-LEVEL QUALITY MEASUREMENTS
# =========================================================================
WORK_SIDE = 1000   # fixed measurement scale: thresholds are only comparable at one size


def _work_gray(img: Image.Image, side: int = WORK_SIDE) -> np.ndarray:
    """Grayscale working copy at a fixed maximum side."""
    g = img.convert("L")
    if max(g.size) > side:
        s = side / max(g.size)
        g = g.resize((max(1, int(g.width * s)), max(1, int(g.height * s))), Image.BILINEAR)
    return np.asarray(g, dtype=np.uint8)


def estimate_noise_sigma(gray: np.ndarray) -> float:
    """Immerkaer's fast noise estimator: convolution with a Laplacian-like mask.

    Cheap (one 3x3 convolution) and, unlike a plain std, it separates sensor/compression noise
    from legitimate image structure."""
    if cv2 is None or gray.size == 0:
        return 0.0
    h, w = gray.shape
    if h < 5 or w < 5:
        return 0.0
    M = np.array([[1, -2, 1], [-2, 4, -2], [1, -2, 1]], dtype=np.float32)
    conv = cv2.filter2D(gray.astype(np.float32), -1, M)
    sigma = float(np.abs(conv).sum() * math.sqrt(0.5 * math.pi) / (6.0 * (w - 2) * (h - 2)))
    return round(sigma, 2)


def estimate_text_height_px(gray: np.ndarray) -> float:
    """Median height of text-like connected components, in pixels of the working copy.

    This is the resolution metric that actually predicts OCR failure -- far more useful than the
    nominal DPI, which says nothing about how large the glyphs are on the page."""
    if cv2 is None or gray.size == 0:
        return 0.0
    binv = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
    n, _, stats, _ = cv2.connectedComponentsWithStats(binv, connectivity=8)
    if n <= 1:
        return 0.0
    hs = []
    H = gray.shape[0]
    for i in range(1, n):
        x, y, w, h, area = stats[i]
        if 3 <= h <= max(8, H * 0.08) and 1 <= w <= H * 0.15 and area >= 6 and h >= w * 0.2:
            hs.append(h)
    return round(float(np.median(hs)), 1) if len(hs) >= 8 else 0.0


def illumination_uniformity(gray: np.ndarray) -> float:
    """Std of 16x16 block means / 255. High = shadows, vignetting, phone-photo lighting."""
    if gray.size == 0:
        return 0.0
    g = gray.astype(np.float32)
    bh, bw = max(1, g.shape[0] // 16), max(1, g.shape[1] // 16)
    blocks = [g[i:i + bh, j:j + bw].mean()
              for i in range(0, g.shape[0] - bh + 1, bh)
              for j in range(0, g.shape[1] - bw + 1, bw)]
    return round(float(np.std(blocks) / 255.0), 4) if blocks else 0.0


def ink_ratio(gray: np.ndarray) -> float:
    """Fraction of pixels belonging to foreground after Otsu. ~0 means a blank page."""
    if cv2 is None or gray.size == 0:
        return 0.0
    binv = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
    return round(float((binv > 0).mean()), 4)


def is_grayscale_image(img: Image.Image, sample: int = 200) -> bool:
    small = img.convert("RGB").resize((sample, sample), Image.BILINEAR)
    a = np.asarray(small, dtype=np.int16)
    return bool(np.abs(a[:, :, 0] - a[:, :, 1]).mean() < 3 and
                np.abs(a[:, :, 1] - a[:, :, 2]).mean() < 3)

In [ ]:
# =========================================================================
# STAGE 1.2 — analyze_page_quality()
# =========================================================================
def analyze_page_quality(img: Image.Image, render_dpi: Optional[int] = None,
                         cfg: Config = CFG) -> Dict[str, Any]:
    """Measure a page and decide which corrections are justified. Pure CV, no GPU, ~30 ms."""
    W, H = img.size
    gray = _work_gray(img)
    scale_back = max(W, H) / max(gray.shape) if max(gray.shape) else 1.0

    lap_var = float(cv2.Laplacian(gray, cv2.CV_64F).var()) if cv2 is not None else \
        float(np.diff(gray.astype(np.float32), axis=1).var())
    p5, p50, p95 = np.percentile(gray, [5, 50, 95])
    contrast = float((p95 - p5) / 255.0)
    brightness = float(gray.mean() / 255.0)
    noise = estimate_noise_sigma(gray)
    text_h_work = estimate_text_height_px(gray)
    text_h_full = round(text_h_work * scale_back, 1)
    uniformity = illumination_uniformity(gray)
    ink = ink_ratio(gray)
    dark_bg = float((gray < 100).mean())

    # blur_score in [0,1]: 0 = unusable, 1 = crisp. Piecewise-linear between the two thresholds.
    blur_score = float(np.clip((lap_var - cfg.blur_var_poor) /
                               max(1e-6, cfg.blur_var_good - cfg.blur_var_poor), 0.0, 1.0))
    contrast_score = float(np.clip(contrast / cfg.contrast_low, 0.0, 1.0)) if \
        contrast < cfg.contrast_low else 1.0
    noise_score = float(np.clip(1.0 - noise / max(1e-6, 2 * cfg.noise_sigma_high), 0.0, 1.0))
    res_score = float(np.clip(text_h_full / (2 * cfg.min_text_height_px), 0.0, 1.0)) \
        if text_h_full > 0 else 0.3

    q = {
        "width": W, "height": H, "megapixels": round(W * H / 1e6, 2),
        "dpi_estimate": render_dpi,
        "text_height_px": text_h_full,
        "blur_laplacian_var": round(lap_var, 1),
        "blur_score": round(blur_score, 3),
        "contrast_raw": round(contrast, 3),
        "contrast_score": round(contrast_score, 3),
        "brightness_score": round(brightness, 3),
        "noise_sigma": noise,
        "noise_score": round(noise_score, 3),
        "resolution_score": round(res_score, 3),
        "illumination_uniformity": uniformity,
        "ink_ratio": ink,
        "dark_background_ratio": round(dark_bg, 3),
        "is_grayscale": is_grayscale_image(img),
        "is_bitonal": bool(len(np.unique(gray)) < 24),
    }

    # ---- derived decisions: each flag has exactly one trigger, so it is explainable ----
    # Two independent conditions must agree before a page is skipped without a model call:
    # a small ID card on a white A4 page has a genuinely tiny ink ratio, so ink alone is not
    # enough -- if any text-like component was found, the page is never called blank.
    q["is_blank"] = bool(ink < cfg.ink_ratio_blank and text_h_full <= 0)
    q["needs_upscale"] = bool(0 < text_h_full < cfg.min_text_height_px)
    q["needs_contrast_enhancement"] = bool(contrast < cfg.contrast_low or
                                           uniformity > cfg.illum_uniformity_poor)
    q["needs_denoise"] = bool(noise > cfg.noise_sigma_high)
    q["needs_invert"] = bool(dark_bg > cfg.dark_background_ratio and brightness < 0.45)
    q["excessive_contrast"] = bool(contrast > cfg.contrast_excessive and q["is_bitonal"])
    q["needs_deskew"] = None          # filled by Stage 2 (requires the skew estimate)
    q["orientation"] = None           # filled by Stage 2

    # Overall visual clarity: the weakest link dominates, hence the min() term.
    combined = 0.35 * blur_score + 0.25 * res_score + 0.25 * contrast_score + 0.15 * noise_score
    q["visual_clarity"] = round(float(min(combined, 0.5 + 0.5 * min(blur_score, res_score))), 3)
    q["grade"] = ("blank" if q["is_blank"] else
                  "good" if q["visual_clarity"] >= 0.75 else
                  "fair" if q["visual_clarity"] >= 0.45 else "poor")
    return q

## Stage 2 — Orientation and adaptive preprocessing

**Problem.** v1 spent a full VLM call per page just to ask which way was up — a complete image
prefill for four output tokens — and then enhanced the page at full 8.7 MP resolution *before*
downscaling, filtering pixels that were about to be discarded.

**Solution — orientation.** A four-tier cascade:

1. Tesseract OSD when the binary exists (accept above `osd_conf_min`), ~200 ms.
2. Projection-profile axis test: horizontal text lines make `var(row_profile)` far exceed
   `var(col_profile)`; rotated 90° the pattern swaps. This settles 0/180 vs 90/270 reliably, ~15 ms.
3. Baseline-asymmetry test for the 180° ambiguity: in upright Latin, Cyrillic and Arabic, ink mass
   sits above each text band's vertical centre. This is the weakest link (~75–85% on clean text),
   which is why the decision margin is recorded.
4. Only when the margin is below `orientation_margin_min`: a 448 px, 4-token VLM probe (~0.3 s).

**Solution — preprocessing.** Correct order, and each operation fires only on its own trigger:

```
rotate (exact 90°)  →  crop to document region  →  resize to target  →  deskew
                    →  contrast/illumination (only if flagged)
                    →  denoise (only if flagged)  →  upscale (only if glyphs are small)
```

Cropping before resizing is the free lunch: an ID card filling 40% of an A4 scan gains ~1.6×
effective resolution *and* costs fewer visual tokens. Enhancement now runs on ~1.2 MP instead of
8.7 MP.

**Integration.** `plan_preprocessing()` consumes Stage 1's flags; the applied operation list is
stored in the page record so any extracted character can be traced back to the exact pixels.

In [ ]:
# =========================================================================
# STAGE 2.1 — LIGHTWEIGHT ORIENTATION DETECTION (CV first, VLM only to break ties)
# =========================================================================
def _binary_ink(gray: np.ndarray) -> np.ndarray:
    return cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]


def _axis_score(binv: np.ndarray) -> float:
    """>1 means text lines run horizontally (0 or 180 deg); <1 means vertically (90 or 270).

    Horizontal text produces alternating ink/blank rows -> high variance in the row profile,
    while the column profile stays flat. The ratio is scale free."""
    row = binv.sum(axis=1).astype(np.float32)
    col = binv.sum(axis=0).astype(np.float32)
    rv = row.var() / (row.mean() ** 2 + 1e-6)
    cv_ = col.var() / (col.mean() ** 2 + 1e-6)
    return float((rv + 1e-6) / (cv_ + 1e-6))


def _updown_score(binv: np.ndarray) -> float:
    """Ink-mass asymmetry within text bands. Positive = upright, negative = upside down.

    Rationale: capitals and ascenders outnumber descenders in Latin/Cyrillic, and Arabic likewise
    carries most of its mass above the baseline, so the ink centroid of a text band sits ABOVE the
    band's geometric centre when the page is the right way up."""
    row = binv.sum(axis=1).astype(np.float32)
    if row.max() <= 0:
        return 0.0
    thresh = row.mean() + 0.3 * row.std()
    bands, start = [], None
    for i, v in enumerate(row):
        if v > thresh and start is None:
            start = i
        elif v <= thresh and start is not None:
            if i - start >= 3:
                bands.append((start, i))
            start = None
    if start is not None and len(row) - start >= 3:
        bands.append((start, len(row)))
    if len(bands) < 3:
        return 0.0
    scores = []
    for a, b in bands[:60]:
        seg = row[a:b]
        if seg.sum() <= 0:
            continue
        idx = np.arange(len(seg), dtype=np.float32)
        centroid = float((seg * idx).sum() / seg.sum()) / max(1, len(seg) - 1)
        scores.append(0.5 - centroid)        # >0 when mass is in the upper half
    return float(np.mean(scores) * 2.0) if scores else 0.0


def detect_orientation(img: Image.Image, vlm_probe=None,
                       cfg: Config = CFG) -> Dict[str, Any]:
    """Returns {'rotation': 0|90|180|270 (counter-clockwise), 'method', 'margin'}."""
    out = {"rotation": 0, "method": "none", "margin": 1.0, "osd_conf": None}
    if cv2 is None:
        return out

    # --- tier 1: tesseract OSD -------------------------------------------
    if pytesseract is not None:
        try:
            small = img.copy()
            small.thumbnail((1000, 1000))
            osd = pytesseract.image_to_osd(small, output_type=TessOutput.DICT,
                                           config="--psm 0")
            conf = float(osd.get("orientation_conf", 0.0))
            if conf >= cfg.osd_conf_min:
                return {"rotation": int(osd.get("rotate", 0)) % 360, "method": "tesseract_osd",
                        "margin": 1.0, "osd_conf": conf}
            out["osd_conf"] = conf
        except Exception:
            pass

    # --- tier 2: axis test -------------------------------------------------
    gray = _work_gray(img, 800)
    binv = _binary_ink(gray)
    axis = _axis_score(binv)
    horizontal = axis >= 1.0
    candidates = (0, 180) if horizontal else (90, 270)

    # --- tier 3: up/down asymmetry ----------------------------------------
    scored = {}
    for deg in candidates:
        rot = binv if deg == 0 else np.rot90(binv, k=(deg // 90))
        scored[deg] = _updown_score(np.ascontiguousarray(rot))
    best = max(scored, key=lambda d: scored[d])
    margin = abs(scored[candidates[0]] - scored[candidates[1]])
    out.update({"rotation": int(best), "method": "cv_projection",
                "margin": round(float(margin), 4), "axis_score": round(float(axis), 3)})

    # --- tier 4: VLM tie-break, only when the CV margin is a coin flip -----
    if margin < cfg.orientation_margin_min and vlm_probe is not None:
        try:
            deg = vlm_probe(img)
            out.update({"rotation": int(deg) % 360, "method": "vlm_tiebreak"})
        except Exception as exc:
            out["probe_error"] = str(exc)
    return out

In [ ]:
# =========================================================================
# STAGE 2.2 — ADAPTIVE PREPROCESSING
# =========================================================================
def estimate_skew(img: Image.Image, cfg: Config = CFG) -> float:
    """Small-angle skew of the dominant text block, in degrees."""
    if cv2 is None:
        return 0.0
    gray = _work_gray(img, 1000)
    binv = _binary_ink(gray)
    binv = cv2.dilate(binv, cv2.getStructuringElement(cv2.MORPH_RECT, (25, 3)))
    coords = cv2.findNonZero(binv)
    if coords is None or len(coords) < 80:
        return 0.0
    ang = cv2.minAreaRect(coords)[-1]
    if ang < -45:
        ang += 90
    elif ang > 45:
        ang -= 90
    return 0.0 if abs(ang) > cfg.deskew_max_deg else round(float(ang), 2)


def find_document_region(img: Image.Image, min_area_frac: float = 0.18) -> Optional[Tuple[int, int, int, int]]:
    """Bounding box of the document inside the scan, or None when it fills the page.

    Cropping BEFORE resizing raises effective resolution and lowers the visual token count at the
    same time. Conservative by design: a 2% margin is kept and odd shapes are rejected, because
    over-cropping would silently remove an MRZ."""
    if cv2 is None:
        return None
    gray = _work_gray(img, 900)
    h, w = gray.shape
    edges = cv2.Canny(cv2.GaussianBlur(gray, (5, 5), 0), 40, 120)
    edges = cv2.morphologyEx(edges, cv2.MORPH_CLOSE,
                             cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15)))
    cnts, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    x, y, cw, ch = cv2.boundingRect(max(cnts, key=cv2.contourArea))
    frac = (cw * ch) / float(w * h)
    if frac < min_area_frac or frac > 0.92:
        return None                                   # nothing found, or it already fills the page
    if cw < 0.25 * w or ch < 0.15 * h:
        return None                                   # implausible shape: refuse rather than risk
    pad_x, pad_y = int(0.02 * w), int(0.02 * h)
    sx, sy = img.width / w, img.height / h
    return (int(max(0, x - pad_x) * sx), int(max(0, y - pad_y) * sy),
            int(min(w, x + cw + pad_x) * sx), int(min(h, y + ch + pad_y) * sy))


def plan_preprocessing(quality: Dict[str, Any], orientation: Dict[str, Any],
                       skew: float, cfg: Config = CFG) -> List[str]:
    """Decide which operations are justified. Each entry traces to exactly one measurement."""
    plan: List[str] = []
    if orientation.get("rotation"):
        plan.append(f"rotate:{orientation['rotation']}")
    if abs(skew) >= cfg.deskew_min_deg:
        plan.append(f"deskew:{skew:+.2f}")
    if quality.get("needs_invert"):
        plan.append("invert")
    if quality.get("needs_contrast_enhancement"):
        plan.append("clahe")
    if quality.get("illumination_uniformity", 0) > cfg.illum_uniformity_poor:
        plan.append("flatten_illumination")
    if quality.get("needs_denoise"):
        plan.append("denoise")
    if quality.get("needs_upscale"):
        plan.append("upscale")
    return plan


def _apply_clahe(img: Image.Image) -> Image.Image:
    g = np.asarray(img.convert("L"))
    return Image.fromarray(cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(g)).convert("RGB")


def _flatten_illumination(img: Image.Image) -> Image.Image:
    """Divide out a heavily blurred copy: removes shadows and vignetting without touching strokes."""
    g = np.asarray(img.convert("L"), dtype=np.float32)
    bg = cv2.GaussianBlur(g, (0, 0), sigmaX=max(g.shape) / 30.0)
    out = np.clip(g / (bg + 1e-3) * float(np.median(bg)), 0, 255).astype(np.uint8)
    return Image.fromarray(out).convert("RGB")


def _denoise(img: Image.Image) -> Image.Image:
    """Edge-preserving only. No median/Gaussian blur: those eat thin strokes and diacritics."""
    a = np.asarray(img.convert("L"))
    return Image.fromarray(cv2.bilateralFilter(a, d=5, sigmaColor=45, sigmaSpace=45)).convert("RGB")


def _resize_max(img: Image.Image, max_side: int) -> Image.Image:
    if max(img.size) <= max_side:
        return img
    s = max_side / max(img.size)
    return img.resize((max(1, int(img.width * s)), max(1, int(img.height * s))), Image.LANCZOS)


def preprocess_page(img: Image.Image, quality: Dict[str, Any], plan: Sequence[str],
                    max_dimension: Optional[int] = None, do_crop: bool = True,
                    cfg: Config = CFG) -> Tuple[Image.Image, Dict[str, Any]]:
    """Apply the plan in the correct order: geometry -> crop -> resize -> photometry."""
    max_dimension = max_dimension or cfg.target_max_dimension
    applied: List[str] = []
    meta: Dict[str, Any] = {"input_size": list(img.size)}

    for op in plan:                                    # 1. exact rotation (lossless, cheap)
        if op.startswith("rotate:"):
            deg = int(op.split(":")[1]) % 360
            if deg:
                img = img.rotate(deg, expand=True)     # PIL rotates counter-clockwise
                applied.append(op)

    if do_crop and cv2 is not None:                    # 2. crop before scaling
        box = find_document_region(img)
        if box:
            img = img.crop(box)
            applied.append("crop_document_region")
            meta["crop_box"] = list(box)

    if "upscale" in plan:                              # 3. resolution policy
        factor = min(cfg.max_upscale_factor,
                     cfg.min_text_height_px / max(1e-6, quality.get("text_height_px") or 1))
        if factor > 1.05:
            target = int(min(max(img.size) * factor, max_dimension * cfg.max_upscale_factor))
            img = img.resize((int(img.width * target / max(img.size)),
                              int(img.height * target / max(img.size))), Image.LANCZOS)
            applied.append(f"upscale_x{factor:.2f}")
    img = _resize_max(img, max_dimension)
    applied.append(f"resize_max_{max_dimension}")

    for op in plan:                                    # 4. deskew after resize (cheaper, same angle)
        if op.startswith("deskew:"):
            ang = float(op.split(":")[1])
            a = np.asarray(img)
            h, w = a.shape[:2]
            M = cv2.getRotationMatrix2D((w / 2, h / 2), ang, 1.0)
            cos, sin = abs(M[0, 0]), abs(M[0, 1])
            nw, nh = int(h * sin + w * cos), int(h * cos + w * sin)
            M[0, 2] += nw / 2 - w / 2
            M[1, 2] += nh / 2 - h / 2
            img = Image.fromarray(cv2.warpAffine(a, M, (nw, nh), flags=cv2.INTER_CUBIC,
                                                 borderMode=cv2.BORDER_REPLICATE))
            applied.append(op)

    if cv2 is not None:                                # 5. photometry, on the SMALL image
        if "invert" in plan:
            img = Image.fromarray(255 - np.asarray(img.convert("L"))).convert("RGB")
            applied.append("invert")
        if "flatten_illumination" in plan:
            img = _flatten_illumination(img); applied.append("flatten_illumination")
        if "clahe" in plan:
            img = _apply_clahe(img); applied.append("clahe")
        if "denoise" in plan:
            img = _denoise(img); applied.append("denoise")

    meta.update({"applied": applied, "output_size": list(img.size),
                 "approx_visual_tokens": int(img.width * img.height / 784)})
    return img, meta

## Stage 3 — Strict extraction prompt and the Qwen engine

**Problem (prompt).** v1's prompt was ~1 200 tokens of prohibitions, re-prefilled on every call,
and it asked for a value plus a self-graded confidence. A value alone offers nothing to check a
hallucination against, and `high/medium/low` is just another generated token.

**Solution (prompt).** Roughly 450 tokens, and three structural changes that do the real work:

* **Five statuses instead of a confidence word.** `CONFIDENT / UNCERTAIN / NOT_VISIBLE /
  NOT_PRESENT / UNREADABLE`. The critical distinction is `NOT_PRESENT` (a driving licence has no
  MRZ) versus `UNREADABLE` (there is an MRZ but the scan is destroyed). v1 collapsed both into
  "unreadable" and polluted every review queue.
* **Mandatory `evidence`.** The model must reproduce the glyphs it sees, using `?` per illegible
  character. If `evidence` and `value` disagree, the value was reconstructed rather than read —
  the most direct hallucination detector available, and it costs nothing at inference time.
* **`text_type` and `script`.** Handwriting (Part 14) and mixed-script documents (Part 13) are
  reported rather than assumed; both feed confidence caps.

**Problem (inference).** `max_new_tokens=1024`, no stop condition, and — the big one — Qwen3 chat
templates enable thinking by default, so a `<think>` block can burn 400–900 tokens before the JSON
starts. Roughly 60% of your six minutes is decoding.

**Solution (inference).** Every parameter and why (Part 10):

| Parameter | Value | Reason |
|---|---|---|
| `do_sample` | `False` | Sampling is literally the mechanism by which a plausible-but-unseen character gets chosen. Greedy is also reproducible, which compliance requires. |
| `temperature` / `top_p` / `top_k` | unset | Ignored under greedy; leaving them set invites accidental sampling and noisy warnings. |
| `repetition_penalty` | **1.0** | See the warning below. |
| `no_repeat_ngram_size` | **0** | See the warning below. |
| `max_new_tokens` | 384 full / 160 targeted | Sized to the schema: 14 fields × ~20 tokens + structure ≈ 300. 1 024 was ~3× the useful ceiling and let degeneration run for 30 s before hitting the cap. |
| thinking mode | **disabled** | `enable_thinking=False`, plus `<think>` stripping in the parser. Transcription is not a reasoning task; a think block is pure cost and gives the model room to talk itself into a plausible completion. |
| stop condition | brace balance | Ends the call the instant the JSON object closes rather than decoding to the cap. |
| assistant prefill | `{` | Forces the answer to open as JSON: removes preamble tokens and most parse failures. |
| `attn_implementation` | `sdpa` (or `flash_attention_2`) | ~3 000-token prefill per page benefits directly. |
| `max_pixels` | `1280 × 28 × 28` | The correct knob for visual token count on Qwen-VL, applied at the processor rather than guessed via pixel size. |

Two parameters deserve emphasis because the usual advice is wrong here:

> `repetition_penalty` must stay at **exactly 1.0** and `no_repeat_ngram_size` at **0**. Anything
> higher penalises tokens already emitted, which is precisely what corrupts identity data:
> `1980`, `AA1122`, and the `<<<<<` filler of an MRZ. Repetition penalty is the wrong tool for
> degeneration in a transcription task — the right tools are a stop condition and a token budget.

**Integration.** The engine is a process-level singleton (`load_model()` caches it), returns token
logprobs for Stage 5, and exposes `generate_batch()` for Stage 7's page batching.

In [ ]:
# =========================================================================
# STAGE 3.1 — FIELD SCHEMA AND PROMPT BUILDER
# =========================================================================
STATUSES = ("CONFIDENT", "UNCERTAIN", "NOT_VISIBLE", "NOT_PRESENT", "UNREADABLE")
POSITIVE_STATUSES = ("CONFIDENT", "UNCERTAIN")     # only these may carry a value


@dataclass(frozen=True)
class FieldSpec:
    name: str
    hint: str
    critical: bool = False


FIELD_SPECS: "OrderedDict[str, FieldSpec]" = OrderedDict((f.name, f) for f in [
    FieldSpec("document_type",     "passport / national ID card / residence permit / driving licence"),
    FieldSpec("surname",           "family name exactly as printed", critical=True),
    FieldSpec("given_names",       "all given names exactly as printed", critical=True),
    FieldSpec("date_of_birth",     "exactly as printed, keep the document's own format", critical=True),
    FieldSpec("place_of_birth",    "exactly as printed"),
    FieldSpec("nationality",       "exactly as printed"),
    FieldSpec("sex",               "single character or word as printed"),
    FieldSpec("document_number",   "exactly as printed", critical=True),
    FieldSpec("personal_number",   "national/personal number if the document shows one"),
    FieldSpec("issue_date",        "exactly as printed"),
    FieldSpec("expiry_date",       "exactly as printed"),
    FieldSpec("issuing_authority", "exactly as printed"),
    FieldSpec("address",           "only if the document shows an address"),
    FieldSpec("mrz",               "all machine-readable lines, preserve every < character"),
])
ALL_FIELDS = list(FIELD_SPECS)
CRITICAL_FIELDS = [k for k, v in FIELD_SPECS.items() if v.critical]

EXTRACTION_SYSTEM = """You are a forensic transcription engine for identity documents. You read pixels. You do not reason about what a document "should" contain.

ABSOLUTE RULES
- Transcribe only characters you can actually see. Never guess, infer, complete or correct.
- Never use knowledge of name spellings, country formats or check digits to fill a gap.
- Never translate, transliterate or normalise. Preserve the original script exactly (Latin, Arabic, Cyrillic, mixed). Preserve every < in an MRZ.
- If one character is illegible, the value is not readable. Put the readable characters in "evidence" with ? for each illegible character, and set "value" to null.
- Handwriting is not assumed readable. If you cannot read it, say so.
- Output JSON only. No markdown, no commentary, no reasoning.

PER FIELD
"value"     the exact transcription, or null
"status"    CONFIDENT   every character clearly legible
            UNCERTAIN   readable but one or more characters are ambiguous
            NOT_VISIBLE the field exists on this document type but is cut off, covered or outside this page
            NOT_PRESENT this document does not have this field at all
            UNREADABLE  the field is there but the image does not allow reading it
"evidence"  what you literally see, using ? per illegible character, or null
"text_type" printed | handwritten | mixed
"script"    latin | arabic | cyrillic | mixed | other

"value" MUST be null unless status is CONFIDENT or UNCERTAIN.
"evidence" must describe what is on the page. Never write an evidence string you cannot see.

Example of correct behaviour when the number reads AB12?45:
{"document_number": {"value": null, "status": "UNCERTAIN", "evidence": "AB12?45", "text_type": "printed", "script": "latin"}}
Writing "AB12345" there would be a critical failure."""


def build_extraction_prompt(fields: Optional[Sequence[str]] = None,
                            page_number: Optional[int] = None,
                            n_pages: Optional[int] = None,
                            attempt_hint: Optional[str] = None) -> str:
    """User prompt. Restricting `fields` on a retry shortens BOTH the prompt and the output."""
    fields = list(fields) if fields else ALL_FIELDS
    lines = [f'  "{f}": {FIELD_SPECS[f].hint}' for f in fields]
    header = "Transcribe this identity document page."
    if page_number and n_pages:
        header += f" This is page {page_number} of {n_pages}."
    if attempt_hint:
        header += f" {attempt_hint}"
    return (f"""{header}

Return one JSON object with exactly these keys:
{chr(10).join(lines)}

Each key maps to an object with "value", "status", "evidence", "text_type", "script".
A field that does not appear on this page is NOT_VISIBLE. A field this document type does not have is NOT_PRESENT.
Return only the JSON object.""")


ORIENTATION_PROBE_SYSTEM = "You answer with a single number and nothing else."
ORIENTATION_PROBE_USER = ("How many degrees counter-clockwise must this scan be rotated so the "
                          "text is upright? Answer only one of: 0, 90, 180, 270.")

PROMPT_HASHES = {
    "system": hashlib.sha256(EXTRACTION_SYSTEM.encode()).hexdigest()[:16],
    "user_full": hashlib.sha256(build_extraction_prompt().encode()).hexdigest()[:16],
}
print("fields:", len(ALL_FIELDS), "| critical:", CRITICAL_FIELDS)
print("system prompt chars:", len(EXTRACTION_SYSTEM),
      "| full user prompt chars:", len(build_extraction_prompt()))
print("prompt hashes:", PROMPT_HASHES)

In [ ]:
# =========================================================================
# STAGE 3.2 — QWEN ENGINE: loaded ONCE, memory-aware, OOM-recovering
# =========================================================================
import gc


@dataclass
class GenOutput:
    text: str
    n_output_tokens: int
    latency_s: float
    token_texts: List[str] = field(default_factory=list)
    token_logprobs: List[float] = field(default_factory=list)
    truncated: bool = False
    error: Optional[str] = None
    error_kind: Optional[str] = None       # "oom" | "inference" | None
    degraded: Optional[str] = None         # records any emergency downscale


def is_oom_error(exc: BaseException) -> bool:
    """Detect CUDA OOM without depending on a specific torch version's exception class."""
    if torch is not None and hasattr(torch, "cuda") and \
            isinstance(exc, getattr(torch.cuda, "OutOfMemoryError", ())):
        return True
    text = f"{type(exc).__name__}: {exc}".lower()
    return "out of memory" in text or "outofmemory" in text or "cuda error: out of memory" in text


def gpu_memory() -> Dict[str, float]:
    """Per-device totals in GiB, as seen by THIS process."""
    if torch is None or not torch.cuda.is_available():
        return {}
    out = {}
    for i in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(i)
        out[f"gpu{i}_total_gib"] = round(total / 2**30, 1)
        out[f"gpu{i}_free_gib"] = round(free / 2**30, 1)
        out[f"gpu{i}_allocated_gib"] = round(torch.cuda.memory_allocated(i) / 2**30, 1)
        out[f"gpu{i}_reserved_gib"] = round(torch.cuda.memory_reserved(i) / 2**30, 1)
    return out


def release_cuda_cache() -> None:
    gc.collect()
    if torch is not None and torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


class _BraceStop:
    """Stops each sequence as soon as its JSON object closes.

    Without this the model decodes to max_new_tokens on every call."""

    def __init__(self, tokenizer, prompt_lens, open_offset: int = 1):
        self.tok = tokenizer
        self.prompt_lens = prompt_lens
        self.open_offset = open_offset      # the prefilled "{" already counts as one open brace
        self.check_every = 8

    def __call__(self, input_ids, scores, **kw):
        n = input_ids.shape[0]
        done = torch.zeros(n, dtype=torch.bool, device=input_ids.device)
        gen_len = input_ids.shape[1] - self.prompt_lens
        if gen_len < 8 or gen_len % self.check_every:
            return done
        for i in range(n):
            txt = self.tok.decode(input_ids[i, self.prompt_lens:], skip_special_tokens=True)
            depth, in_str, esc = self.open_offset, False, False
            for c in txt:
                if in_str:
                    if esc:        esc = False
                    elif c == "\\": esc = True
                    elif c == '"': in_str = False
                    continue
                if c == '"':   in_str = True
                elif c == "{": depth += 1
                elif c == "}":
                    depth -= 1
                    if depth <= 0:
                        done[i] = True
                        break
        return done


class _LogprobRecorder:
    """Records the log-probability of each chosen token, as a LogitsProcessor.

    Why not `output_scores=True`: that keeps one [batch, vocab] tensor PER STEP for the whole
    generation. At a ~150k vocab and 384 steps that is ~115 MB per sequence of pure waste, held
    exactly when memory is tightest. Under greedy decoding the chosen token is the argmax, so one
    scalar per step is all the information we actually use."""

    def __init__(self, batch_size: int):
        self.token_ids: List[List[int]] = [[] for _ in range(batch_size)]
        self.logprobs: List[List[float]] = [[] for _ in range(batch_size)]

    def __call__(self, input_ids, scores):
        lp = torch.log_softmax(scores.float(), dim=-1)
        top = lp.argmax(dim=-1)
        vals = lp.gather(1, top.unsqueeze(1)).squeeze(1)
        top_c, vals_c = top.tolist(), vals.tolist()
        for i in range(len(top_c)):
            if i < len(self.token_ids):
                self.token_ids[i].append(int(top_c[i]))
                self.logprobs[i].append(float(vals_c[i]))
        return scores


class QwenVLEngine:
    """Process-level singleton. The model is loaded once per Python process, never per document."""
    _instance: Optional["QwenVLEngine"] = None

    def __init__(self, cfg: Config = CFG):
        from transformers import AutoConfig, AutoProcessor
        self.cfg = cfg
        assert torch is not None, "PyTorch required"
        assert Path(cfg.model_path).exists(), f"model path not found: {cfg.model_path}"

        # ---- memory preflight ------------------------------------------------
        # The most common cause of "OOM on every call" in a notebook is a SECOND copy of the
        # model already resident because a load cell was re-run. Say so loudly before loading.
        before = gpu_memory()
        if before:
            log.info("GPU before load: %s", before)
            busy = max((v for k, v in before.items() if k.endswith("allocated_gib")), default=0.0)
            if busy > 1.0:
                log.warning("%.1f GiB already allocated on GPU by this process. If you re-ran the "
                            "model cell, call free_model() or restart the kernel first: two copies "
                            "of the weights will OOM every generate().", busy)

        self.hf_config = AutoConfig.from_pretrained(cfg.model_path, trust_remote_code=True,
                                                    local_files_only=True)
        arch = " ".join(getattr(self.hf_config, "architectures", []) or []).lower()
        if not (hasattr(self.hf_config, "vision_config") or
                any(h in arch for h in ("vl", "vision", "image_text"))):
            raise RuntimeError(
                f"{cfg.model_path} does not expose a vision tower "
                f"(architectures={getattr(self.hf_config, 'architectures', None)}). "
                "Image transcription needs the VL checkpoint.")

        # max_pixels is THE knob for visual token count on Qwen-VL: one token ~= a 28x28 block.
        px = cfg.max_visual_tokens * 28 * 28
        try:
            self.processor = AutoProcessor.from_pretrained(
                cfg.model_path, trust_remote_code=True, local_files_only=True,
                min_pixels=256 * 28 * 28, max_pixels=px)
        except TypeError:
            self.processor = AutoProcessor.from_pretrained(cfg.model_path, trust_remote_code=True,
                                                           local_files_only=True)
        self.tokenizer = getattr(self.processor, "tokenizer", self.processor)
        try:
            self.tokenizer.padding_side = "left"      # required for correct batched generation
        except Exception:
            pass

        kwargs = dict(torch_dtype=getattr(torch, cfg.torch_dtype), device_map=cfg.device_map,
                      trust_remote_code=True, local_files_only=True, low_cpu_mem_usage=True)
        if cfg.attn_impl:
            kwargs["attn_implementation"] = cfg.attn_impl

        # ---- reserve headroom for activations --------------------------------
        # With device_map="auto" and no budget, accelerate happily fills the card with WEIGHTS,
        # leaving nothing for the vision encoder, the KV cache and the logits tensor -- which is
        # exactly the "weights loaded fine, every generate OOMs" failure. Reserving headroom lets
        # accelerate spill the overflow layers to CPU instead: slower, but it runs.
        if cfg.reserve_vram_gib > 0 and torch.cuda.is_available() and cfg.device_map == "auto":
            mm = {}
            for i in range(torch.cuda.device_count()):
                _, total = torch.cuda.mem_get_info(i)
                budget = max(1, int(total / 2**30) - int(cfg.reserve_vram_gib))
                mm[i] = f"{budget}GiB"
            mm["cpu"] = f"{int(cfg.cpu_offload_gib)}GiB"
            kwargs["max_memory"] = mm
            log.info("max_memory budget: %s (reserving %.0f GiB for activations)",
                     mm, cfg.reserve_vram_gib)

        import transformers as tf
        last, self.model = None, None
        for cls_name in list(getattr(self.hf_config, "architectures", []) or []) + \
                        ["AutoModelForImageTextToText", "AutoModelForVision2Seq"]:
            if not hasattr(tf, cls_name):
                continue
            try:
                self.model = getattr(tf, cls_name).from_pretrained(cfg.model_path, **kwargs)
                self.loader = cls_name
                break
            except Exception as exc:
                last = f"{cls_name}: {type(exc).__name__}: {exc}"
                release_cuda_cache()
        if self.model is None:
            raise RuntimeError(f"could not load model. last error: {last}")

        self.model.eval()
        gc_ = getattr(self.model, "generation_config", None)
        if gc_ is not None:                            # kill any sampling defaults in the checkpoint
            gc_.do_sample = False
            gc_.temperature = gc_.top_p = gc_.top_k = None
        self.name = f"{Path(cfg.model_path).parent.name} [{self.loader}]"
        self.consecutive_ooms = 0
        after = gpu_memory()
        log.info("model loaded: %s", self.name)
        if after:
            log.info("GPU after load: %s", after)
            free = min((v for k, v in after.items() if k.endswith("free_gib")), default=99.0)
            if free < 6.0:
                log.warning("only %.1f GiB free after loading weights. Inference needs headroom "
                            "for the vision encoder and KV cache -- expect OOM. Raise "
                            "CFG.reserve_vram_gib, lower CFG.max_visual_tokens, or free the GPU.",
                            free)

    # ------------------------------------------------------------------
    @classmethod
    def get(cls, cfg: Config = CFG) -> "QwenVLEngine":
        """Return the cached engine, or build one. Never loads a second copy silently."""
        inst = cls._instance
        if inst is not None:
            if inst.cfg.model_path == cfg.model_path:
                inst.cfg = cfg                          # config may be re-tuned between runs
                return inst
            log.warning("model path changed; freeing the previous model first")
            free_model()
        cls._instance = cls(cfg)
        return cls._instance

    def _template(self, system: str, user: str) -> str:
        messages = [{"role": "system", "content": [{"type": "text", "text": system}]},
                    {"role": "user", "content": [{"type": "image"},
                                                 {"type": "text", "text": user}]}]
        try:    # Qwen3 thinking mode is ON by default and would burn hundreds of tokens per page
            text = self.processor.apply_chat_template(messages, tokenize=False,
                                                      add_generation_prompt=True,
                                                      enable_thinking=False)
        except TypeError:
            text = self.processor.apply_chat_template(messages, tokenize=False,
                                                      add_generation_prompt=True)
        return text

    # ------------------------------------------------------------------
    def _generate_once(self, images: Sequence[Image.Image], system: str,
                       users: Sequence[str], mnt: int) -> List[GenOutput]:
        """One raw generate. Raises on OOM; recovery is handled by the caller."""
        from transformers import StoppingCriteriaList, LogitsProcessorList
        cfg = self.cfg
        texts = [self._template(system, u) + cfg.json_prefill for u in users]
        inputs = self.processor(text=texts, images=list(images), padding=True,
                                return_tensors="pt").to(self.model.device)
        prompt_len = inputs["input_ids"].shape[1]
        recorder = _LogprobRecorder(len(users))
        stopper = StoppingCriteriaList([_BraceStop(self.tokenizer, prompt_len,
                                                   open_offset=len(cfg.json_prefill))])
        t0 = time.perf_counter()
        with torch.inference_mode():
            out = self.model.generate(
                **inputs, max_new_tokens=mnt,
                do_sample=False, num_beams=1,
                repetition_penalty=cfg.repetition_penalty,      # pinned to 1.0 on purpose
                no_repeat_ngram_size=cfg.no_repeat_ngram_size,  # pinned to 0 on purpose
                stopping_criteria=stopper,
                logits_processor=LogitsProcessorList([recorder]),
                return_dict_in_generate=True,
                output_scores=False,            # the recorder already has what we need
                pad_token_id=getattr(self.tokenizer, "pad_token_id", None)
                             or getattr(self.tokenizer, "eos_token_id", None))
        latency = time.perf_counter() - t0

        seqs = out.sequences[:, prompt_len:].detach().to("cpu")
        del out, inputs
        results: List[GenOutput] = []
        pad = getattr(self.tokenizer, "pad_token_id", None)
        for i in range(seqs.shape[0]):
            ids = seqs[i]
            if pad is not None:
                keep = ids != pad
                ids = ids[keep] if keep.any() else ids
            n = int(ids.shape[0])
            text = cfg.json_prefill + self.tokenizer.decode(ids, skip_special_tokens=True)
            toks = [self.tokenizer.decode([t]) for t in recorder.token_ids[i][:n]]
            results.append(GenOutput(text=text, n_output_tokens=n,
                                     latency_s=round(latency / max(1, seqs.shape[0]), 3),
                                     token_texts=toks,
                                     token_logprobs=recorder.logprobs[i][:n],
                                     truncated=bool(n >= mnt)))
        return results

    def generate_batch(self, images: Sequence[Image.Image], system: str, users: Sequence[str],
                       max_new_tokens: Optional[int] = None,
                       want_logprobs: bool = True, _depth: int = 0) -> List[GenOutput]:
        """Generate with graded OOM recovery: split the batch, then shrink the image, then stop."""
        cfg = self.cfg
        mnt = max_new_tokens or cfg.max_new_tokens
        images = list(images)
        try:
            res = self._generate_once(images, system, users, mnt)
            self.consecutive_ooms = 0
            return res
        except Exception as exc:
            if not is_oom_error(exc):
                raise
            release_cuda_cache()
            self.consecutive_ooms += 1

            # 1. a batch is the cheapest thing to give up: halve it
            if len(images) > 1:
                log.warning("OOM on a batch of %d; splitting", len(images))
                mid = len(images) // 2
                return (self.generate_batch(images[:mid], system, users[:mid], mnt,
                                            want_logprobs, _depth) +
                        self.generate_batch(images[mid:], system, users[mid:], mnt,
                                            want_logprobs, _depth))

            # 2. single image: shrink it and try again. This DEGRADES the read, so it is
            #    recorded on the result and feeds straight into the confidence score.
            if _depth < cfg.oom_max_downscales:
                factor = cfg.oom_downscale_factor
                small = images[0].resize((max(64, int(images[0].width * factor)),
                                          max(64, int(images[0].height * factor))),
                                         Image.LANCZOS)
                log.warning("OOM on a single page; retrying at %d%% (%dx%d)",
                            int(factor * 100), small.width, small.height)
                outs = self.generate_batch([small], system, users, mnt, want_logprobs, _depth + 1)
                for o in outs:
                    o.degraded = f"oom_downscale_x{factor ** (_depth + 1):.2f}"
                return outs

            # 3. out of options: report the failure honestly, do not pretend the page was read
            log.error("OOM persists after %d downscales; giving up on this page",
                      cfg.oom_max_downscales)
            return [GenOutput(text="", n_output_tokens=0, latency_s=0.0,
                              error=f"{type(exc).__name__}: {exc}", error_kind="oom")
                    for _ in users]

    def generate(self, image: Image.Image, system: str, user: str, **kw) -> GenOutput:
        return self.generate_batch([image], system, [user], **kw)[0]

    def orientation_probe(self, img: Image.Image) -> int:
        probe = img.copy()
        probe.thumbnail((448, 448))                    # deliberately tiny: this is a 4-token call
        r = self.generate(probe, ORIENTATION_PROBE_SYSTEM, ORIENTATION_PROBE_USER,
                          max_new_tokens=self.cfg.max_new_tokens_probe, want_logprobs=False)
        m = re.search(r"\b(0|90|180|270)\b", r.text or "")
        return int(m.group(1)) if m else 0


class MockEngine:
    """Rehearsal without a GPU. Returns a syntactically valid all-null page: invents nothing."""
    name = "MOCK"
    consecutive_ooms = 0

    def generate_batch(self, images, system, users, max_new_tokens=None, want_logprobs=True, **kw):
        body = {f: {"value": None, "status": "UNREADABLE", "evidence": None,
                    "text_type": "printed", "script": "latin"} for f in ALL_FIELDS}
        txt = json.dumps(body)
        return [GenOutput(text=txt, n_output_tokens=len(txt) // 4, latency_s=0.01)
                for _ in users]

    def generate(self, image, system, user, **kw):
        return self.generate_batch([image], system, [user])[0]

    def orientation_probe(self, img):
        return 0


def free_model() -> None:
    """Drop the loaded model and return its VRAM. Call this before re-loading in the same kernel."""
    inst = QwenVLEngine._instance
    if inst is not None:
        try:
            inst.model.to("meta")
        except Exception:
            pass
        del inst.model
        QwenVLEngine._instance = None
    release_cuda_cache()
    log.info("model freed. GPU now: %s", gpu_memory() or "no CUDA")


def load_model(cfg: Config = CFG):
    """Load the model ONCE. Every document and page reuses this instance."""
    if cfg.mock_model:
        return MockEngine()
    return QwenVLEngine.get(cfg)


ENGINE = load_model(CFG)
log.info("engine ready: %s", ENGINE.name)

### 3.3 Diagnosing and preventing CUDA OOM

If `generate()` fails instantly with `memory allocation failed with OOM` and the run log shows
`tokens=0`, the model never produced anything — this is an infrastructure failure, **not** an
unreadable document. Causes, in the order worth checking:

| # | Cause | Check | Fix |
|---|---|---|---|
| 1 | A second copy of the model is resident because a load cell was re-run | `gpu_report()` — allocated is roughly 2× the weight size | `free_model()`, or restart the kernel |
| 2 | Another process shares the GPU | `gpu_report()` lists other PIDs | wait, or pin a device with `CUDA_VISIBLE_DEVICES` |
| 3 | `device_map="auto"` filled the card with weights, leaving nothing for activations | free VRAM after load is under ~6 GiB | raise `CFG.reserve_vram_gib` (spills layers to CPU: slower, but it runs) |
| 4 | Batch too large for the headroom | OOM only when `page_batch_size > 1` | `autotune_batch_size()` |
| 5 | Images too large | OOM on big pages only | lower `CFG.max_visual_tokens` / `target_max_dimension` |
| 6 | Fragmentation | plenty free yet still OOM | `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True`, already set in cell 0.1 — needs a kernel restart to take effect |

The pipeline now degrades in three graded steps rather than failing outright: split the batch,
shrink the image (recorded as `oom_downscale`, and it caps the field confidence because the read
came from fewer pixels), then report `error_kind="oom"` and stop.

**A page that OOMs is never reported as `UNREADABLE`.** It gets the document status `ERROR`, which
routes to manual review. Conflating "our GPU ran out of memory" with "this scan is illegible"
would put a false quality signal into the KYC record.

In [ ]:
# =========================================================================
# STAGE 3.3 — GPU DIAGNOSTICS AND BATCH AUTOTUNING
# =========================================================================
import subprocess


def gpu_report() -> None:
    """Print this process's memory AND any other process holding the GPU."""
    if torch is None or not torch.cuda.is_available():
        print("no CUDA device visible")
        return
    for k, v in gpu_memory().items():
        print(f"{k:<24} {v}")
    print("-" * 60)
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-compute-apps=pid,process_name,used_memory",
             "--format=csv,noheader"],
            capture_output=True, text=True, timeout=15).stdout.strip()
        print("processes on the GPU (a second PID here explains most OOMs):")
        print(out or "  (none reported)")
        print(f"this process pid = {os.getpid()}")
    except Exception as exc:
        print("nvidia-smi unavailable:", exc)


def autotune_batch_size(engine=None, sample_image: Optional[Image.Image] = None,
                        candidates: Sequence[int] = (1, 2, 4, 8),
                        cfg: Config = CFG) -> int:
    """Find the largest page batch that fits, by measuring instead of guessing.

    Run this once per environment, then set CFG.page_batch_size to the result. Re-run it if the
    model, the image budget or the GPU allocation changes."""
    engine = engine or ENGINE
    if sample_image is None:
        side = cfg.target_max_dimension
        sample_image = Image.new("RGB", (side, int(side * 0.7)), "white")
    prompt = build_extraction_prompt(ALL_FIELDS, 1, 1)
    best = 1
    for n in candidates:
        release_cuda_cache()
        if torch is not None and torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
        try:
            t0 = time.perf_counter()
            outs = engine.generate_batch([sample_image] * n, EXTRACTION_SYSTEM, [prompt] * n,
                                         max_new_tokens=64)
            dt = time.perf_counter() - t0
            if any(o.error_kind == "oom" for o in outs):
                print(f"batch {n:>2}: OOM")
                break
            peak = (torch.cuda.max_memory_allocated() / 2**30
                    if torch is not None and torch.cuda.is_available() else 0.0)
            print(f"batch {n:>2}: ok  {dt:5.1f}s total  {dt/n:5.2f}s/page  peak {peak:5.1f} GiB")
            best = n
        except Exception as exc:
            print(f"batch {n:>2}: failed ({type(exc).__name__})")
            break
    release_cuda_cache()
    print(f"\nrecommended: CFG.page_batch_size = {best}")
    return best


gpu_report()
# CFG.page_batch_size = autotune_batch_size()   # run once, then pin the value in the config cell

## Stage 4 — Parsing, coercion and field validation

**Problem.** v1 accepted whatever JSON came back. A value with a contradictory status, evidence
that did not match the value, an impossible date, an expiry before an issue date — none of it was
caught, so a confidently-wrong value looked exactly like a correct one.

**Solution.** Three layers, all of which only ever *remove* trust:

1. **Parse, fail closed.** Strip any stray `<think>` block, take the first balanced object, and on
   failure return an all-`UNREADABLE` page rather than a partial guess.
2. **Coerce contradictions.** A value carried alongside `UNREADABLE`/`NOT_VISIBLE`/`NOT_PRESENT` is
   nulled. Evidence containing `?` forces at most `UNCERTAIN`. Evidence that does not reproduce the
   value raises `evidence_mismatch`.
3. **Validate, never repair.** Each validator returns `valid / invalid / unknown` plus flags. The
   `O`/`0` homoglyph case from Part 7 produces `NEEDS_REVIEW` — it never rewrites the character,
   because the only thing that could justify a substitution is the pixels, and the validator cannot
   see them.

**Integration.** Validator output becomes the `format_validity` and `consistency` components of the
Stage 5 score, and any raised flag is carried into the review queue.

In [ ]:
# =========================================================================
# STAGE 4.1 — PARSING AND COERCION (fail closed)
# =========================================================================
NULLISH = {"", "null", "none", "n/a", "na", "unknown", "-", "--", "?"}


def blank_field(status: str = "UNREADABLE", reason: Optional[str] = None) -> Dict[str, Any]:
    return {"value": None, "status": status, "evidence": None, "text_type": None,
            "script": None, "flags": ([reason] if reason else [])}


def blank_page_fields(fields: Sequence[str] = None, reason: str = None) -> Dict[str, Any]:
    return {f: blank_field(reason=reason) for f in (fields or ALL_FIELDS)}


def _first_json_object(text: str) -> Optional[str]:
    if not text:
        return None
    t = re.sub(r"<think>.*?</think>", "", text, flags=re.S)      # belt and braces
    t = re.sub(r"^```[a-zA-Z]*\s*|\s*```$", "", t.strip())
    start = t.find("{")
    if start < 0:
        return None
    depth, in_str, esc = 0, False, False
    for i in range(start, len(t)):
        c = t[i]
        if in_str:
            if esc:        esc = False
            elif c == "\\": esc = True
            elif c == '"': in_str = False
            continue
        if c == '"':   in_str = True
        elif c == "{": depth += 1
        elif c == "}":
            depth -= 1
            if depth == 0:
                return t[start:i + 1]
    return t[start:] + "}" * depth if depth > 0 else None        # tolerate a truncated tail


def _clean_scalar(v: Any) -> Optional[str]:
    if isinstance(v, (list, tuple)):
        v = "\n".join(str(x) for x in v if x is not None)
    if v is None or isinstance(v, bool):
        return None
    if isinstance(v, (int, float)):
        v = str(v)
    if not isinstance(v, str):
        return None
    s = v.strip()
    return None if s.lower() in NULLISH else s


def coerce_field(node: Any, name: str) -> Dict[str, Any]:
    """Map one model field onto the schema. Only ever reduces trust, never raises it."""
    flags: List[str] = []
    if not isinstance(node, dict):
        value, status, evidence, ttype, script = _clean_scalar(node), None, None, None, None
        if value is not None:
            flags.append("unstructured_field")
    else:
        value = _clean_scalar(node.get("value"))
        evidence = _clean_scalar(node.get("evidence"))
        status = str(node.get("status", "")).strip().upper() or None
        ttype = (str(node.get("text_type", "")).strip().lower() or None)
        script = (str(node.get("script", "")).strip().lower() or None)

    if status not in STATUSES:
        if status:
            flags.append(f"bad_status:{status}")
        status = "CONFIDENT" if value is not None else "UNREADABLE"
        flags.append("status_inferred")

    # a value may only exist under a positive status
    if value is not None and status not in POSITIVE_STATUSES:
        flags.append("value_with_negative_status_dropped")
        value = None
    if value is None and status in POSITIVE_STATUSES:
        status = "UNREADABLE"
        flags.append("positive_status_without_value")

    # evidence rules
    if evidence and "?" in evidence:
        if status == "CONFIDENT":
            status = "UNCERTAIN"
            flags.append("downgraded_illegible_evidence")
        if value is not None:
            flags.append("value_despite_illegible_evidence")     # strong hallucination signal
    if value is not None and not evidence:
        flags.append("missing_evidence")
    if value is not None and evidence:
        ratio = difflib.SequenceMatcher(None, compare_key(value), compare_key(evidence)).ratio()
        if ratio < 0.85:
            flags.append("evidence_mismatch")
    if ttype not in (None, "printed", "handwritten", "mixed"):
        flags.append("bad_text_type")
        ttype = None

    return {"value": value, "status": status, "evidence": evidence,
            "text_type": ttype, "script": script, "flags": flags}


def parse_page_output(raw_text: str, fields: Sequence[str] = None) -> Tuple[Dict[str, Any], bool, Optional[str]]:
    fields = list(fields or ALL_FIELDS)
    block = _first_json_object(raw_text)
    if block is None:
        return blank_page_fields(fields, "no_json_in_output"), False, "no_json"
    try:
        obj = json.loads(block)
    except json.JSONDecodeError as exc:
        try:
            obj = json.loads(re.sub(r",\s*([}\]])", r"\1", block))
        except Exception:
            return blank_page_fields(fields, "unparsable_json"), False, f"json_error: {exc}"
    if not isinstance(obj, dict):
        return blank_page_fields(fields, "not_an_object"), False, "not_an_object"

    lower = {str(k).strip().lower(): k for k in obj}
    out = {}
    for f in fields:
        key = lower.get(f) or (lower.get("machine_readable_zone") if f == "mrz" else None)
        out[f] = coerce_field(obj[key], f) if key else blank_field("NOT_VISIBLE", "field_absent_from_output")
    extra = [k for k in obj if k not in {lower.get(f) for f in fields}]
    return out, True, (f"extra_keys:{','.join(map(str, extra))[:120]}" if extra else None)

In [ ]:
# =========================================================================
# STAGE 4.2 — FIELD VALIDATORS (flag only; never modify the model's answer)
# =========================================================================
DATE_PATTERNS = [
    (re.compile(r"^(\d{2})[./\- ](\d{2})[./\- ](\d{4})$"), ("d", "m", "y")),
    (re.compile(r"^(\d{4})[./\- ](\d{2})[./\- ](\d{2})$"), ("y", "m", "d")),
    (re.compile(r"^(\d{2})[./\- ](\d{2})[./\- ](\d{2})$"), ("d", "m", "yy")),
    (re.compile(r"^(\d{2})\s?([A-Z]{3})\s?(\d{4})$"),      ("d", "mon", "y")),
]
MONTHS3 = {m: i + 1 for i, m in enumerate(
    ["JAN", "FEB", "MAR", "APR", "MAY", "JUN", "JUL", "AUG", "SEP", "OCT", "NOV", "DEC"])}
HOMOGLYPHS = {"O": "0", "0": "O", "I": "1", "1": "I", "L": "1", "S": "5",
              "5": "S", "B": "8", "8": "B", "Z": "2", "2": "Z", "G": "6", "6": "G"}
ISO3_SAMPLE = {"FRA", "MAR", "DZA", "TUN", "USA", "GBR", "DEU", "ESP", "ITA", "PRT", "BEL",
               "NLD", "CHE", "RUS", "UKR", "TUR", "EGY", "SEN", "CIV", "MLI", "CMR", "COD",
               "NGA", "CHN", "IND", "BRA", "CAN", "SYR", "LBN", "IRQ", "IRN", "SAU", "ARE"}


@dataclass
class Validation:
    valid: Optional[bool] = None            # None = not checkable, not "invalid"
    flags: List[str] = field(default_factory=list)
    parsed: Optional[str] = None            # normalised copy for CROSS-CHECKS ONLY, never output


def parse_date(value: str) -> Optional[date]:
    """Best-effort parse for consistency checks. The stored value is never replaced by this."""
    s = str(value).strip().upper()
    for rx, order in DATE_PATTERNS:
        m = rx.match(s)
        if not m:
            continue
        g = dict(zip(order, m.groups()))
        try:
            y = int(g.get("y") or (2000 + int(g["yy"]) if int(g["yy"]) < 30 else 1900 + int(g["yy"])))
            mth = MONTHS3.get(g.get("mon"), 0) or int(g.get("m", 0))
            return date(y, mth, int(g["d"]))
        except Exception:
            return None
    return None


def validate_date(value: str, kind: str = "generic") -> Validation:
    v = Validation()
    d = parse_date(value)
    if d is None:
        v.valid = False
        v.flags.append("unrecognised_date_format")
        return v
    v.parsed = d.isoformat()
    today = date.today()
    if kind == "birth":
        age = (today - d).days / 365.25
        v.valid = 0 <= age <= 120
        if not v.valid:
            v.flags.append(f"implausible_age:{age:.0f}")
    elif kind == "issue":
        v.valid = date(1950, 1, 1) <= d <= today
        if d > today:
            v.flags.append("issue_date_in_future")
    elif kind == "expiry":
        v.valid = date(1950, 1, 1) <= d <= date(today.year + 25, 12, 31)
        if d < today:
            v.flags.append("document_expired")          # informational, not an error
    else:
        v.valid = True
    return v


def validate_document_number(value: str) -> Validation:
    v = Validation()
    s = str(value).strip()
    if not re.fullmatch(r"[A-Z0-9<\-/ ]{5,20}", s.upper()):
        v.valid = False
        v.flags.append("unexpected_document_number_charset")
        return v
    v.valid = True
    core = s.upper().replace(" ", "")
    # Homoglyph risk: FLAG ONLY. Substituting here would be exactly the guessing we forbid.
    if any(c in HOMOGLYPHS for c in core) and re.search(r"\d", core) and re.search(r"[A-Z]", core):
        risky = sorted({c for c in core if c in HOMOGLYPHS})
        v.flags.append("homoglyph_risk:" + "".join(risky))
    if len(set(core)) <= 2:
        v.flags.append("suspiciously_uniform")
    return v


def validate_name(value: str) -> Validation:
    v = Validation()
    s = str(value).strip()
    if re.search(r"\d", s):
        v.valid = False
        v.flags.append("digits_in_name")
        return v
    if len(s) < 2:
        v.valid = False
        v.flags.append("name_too_short")
        return v
    scripts = set()
    for ch in s:
        if ch.isspace() or ch in "-'.,":
            continue
        try:
            blk = unicodedata.name(ch, "")
        except Exception:
            blk = ""
        scripts.add("arabic" if "ARABIC" in blk else "cyrillic" if "CYRILLIC" in blk
                    else "latin" if "LATIN" in blk else "other")
    v.valid = bool(scripts) and scripts <= {"latin", "arabic", "cyrillic"}
    if "other" in scripts:
        v.flags.append("unexpected_characters_in_name")
    if len(scripts) > 1:
        v.flags.append("mixed_script_name")            # legitimate on bilingual documents
    return v


def validate_sex(value: str) -> Validation:
    v = Validation()
    s = str(value).strip().upper()
    v.valid = s in {"M", "F", "X", "H", "MALE", "FEMALE", "MASCULIN", "FEMININ", "FÉMININ",
                    "ذكر", "أنثى"}
    if not v.valid:
        v.flags.append("unexpected_sex_value")
    return v


def validate_nationality(value: str) -> Validation:
    v = Validation()
    s = str(value).strip().upper()
    if re.fullmatch(r"[A-Z]{3}", s):
        v.valid = s in ISO3_SAMPLE or None
        if v.valid is None:
            v.flags.append("iso3_not_in_local_list")   # not an error: the list is a sample
    else:
        v.valid = bool(re.fullmatch(r"[^\d]{3,40}", s))
        if not v.valid:
            v.flags.append("unexpected_nationality_value")
    return v


def _mrz_check_digit(s: str) -> Optional[str]:
    w, total = (7, 3, 1), 0
    for i, c in enumerate(s):
        if c == "<":       val = 0
        elif c.isdigit():  val = int(c)
        elif c.isalpha():  val = ord(c.upper()) - 55
        else:              return None
        total += val * w[i % 3]
    return str(total % 10)


def validate_mrz(value: str) -> Validation:
    """ICAO 9303 check digits. Reporting only: a mismatch NEVER rewrites a character."""
    v = Validation()
    lines = [l.strip().replace(" ", "") for l in str(value).splitlines() if l.strip()]
    if not lines:
        v.valid = None
        return v
    fmt = ("TD3" if len(lines) == 2 and max(map(len, lines)) >= 43 else
           "TD2" if len(lines) == 2 else "TD1" if len(lines) == 3 else "unknown")
    checks = {}
    try:
        if fmt == "TD3" and len(lines[1]) >= 28:
            l2 = lines[1]
            for name, (a, b, cd) in {"document_number": (0, 9, 9), "date_of_birth": (13, 19, 19),
                                     "expiry_date": (21, 27, 27)}.items():
                checks[name] = (_mrz_check_digit(l2[a:b]), l2[cd])
        elif fmt == "TD1" and len(lines[0]) >= 15 and len(lines[1]) >= 15:
            checks["document_number"] = (_mrz_check_digit(lines[0][5:14]), lines[0][14])
            checks["date_of_birth"] = (_mrz_check_digit(lines[1][0:6]), lines[1][6])
            checks["expiry_date"] = (_mrz_check_digit(lines[1][8:14]), lines[1][14])
    except Exception:
        v.flags.append("mrz_parse_error")
    if not checks:
        v.valid = None
        v.flags.append(f"mrz_not_verifiable:{fmt}")
        return v
    ok = [c == p for c, p in checks.values() if c is not None]
    v.valid = bool(ok) and all(ok)
    if not v.valid:
        bad = [k for k, (c, p) in checks.items() if c is not None and c != p]
        v.flags.append("mrz_checksum_mismatch:" + ",".join(bad))
    v.parsed = fmt
    return v


VALIDATORS = {
    "date_of_birth": lambda v: validate_date(v, "birth"),
    "issue_date":    lambda v: validate_date(v, "issue"),
    "expiry_date":   lambda v: validate_date(v, "expiry"),
    "document_number": validate_document_number,
    "personal_number": validate_document_number,
    "surname":       validate_name,
    "given_names":   validate_name,
    "place_of_birth": validate_name,
    "issuing_authority": lambda v: Validation(valid=bool(str(v).strip())),
    "sex":           validate_sex,
    "nationality":   validate_nationality,
    "mrz":           validate_mrz,
}


def validate_extraction(fields: Dict[str, Any]) -> Dict[str, Any]:
    """Per-field format validation + cross-field consistency. Returns findings only."""
    result: Dict[str, Any] = {"fields": {}, "cross_field": {"flags": [], "checks": {}}}
    for name, node in fields.items():
        val = node.get("value")
        if val is None:
            result["fields"][name] = {"valid": None, "flags": []}
            continue
        v = VALIDATORS.get(name, lambda x: Validation(valid=True))(val)
        result["fields"][name] = {"valid": v.valid, "flags": list(v.flags), "parsed": v.parsed}

    # ---- cross-field consistency ----
    cf = result["cross_field"]
    dob = parse_date(fields.get("date_of_birth", {}).get("value") or "")
    iss = parse_date(fields.get("issue_date", {}).get("value") or "")
    exp = parse_date(fields.get("expiry_date", {}).get("value") or "")
    if dob and iss:
        cf["checks"]["birth_before_issue"] = dob < iss
        if dob >= iss:
            cf["flags"].append("birth_after_issue")
    if iss and exp:
        cf["checks"]["issue_before_expiry"] = iss < exp
        if iss >= exp:
            cf["flags"].append("issue_after_expiry")
    if dob and exp:
        cf["checks"]["age_at_expiry_plausible"] = 0 < (exp - dob).days / 365.25 < 130

    # ---- MRZ vs visually-read fields: agreement is real evidence, disagreement is a red flag ----
    mrz = fields.get("mrz", {}).get("value")
    if mrz:
        flat = compare_key(mrz)
        for f in ("document_number", "surname"):
            v = fields.get(f, {}).get("value")
            if v:
                agree = compare_key(v) in flat
                cf["checks"][f"mrz_contains_{f}"] = agree
                if not agree:
                    cf["flags"].append(f"mrz_disagrees_with_{f}")
    return result

## Stage 5 — Confidence you can defend

**Problem.** v1 reported whatever confidence word the model generated. That number could not be
audited, could not be calibrated, and correlated with fluency rather than with legibility.

**Solution.** The model's self-report is demoted to a *gate*. The score is a weighted sum of five
independently measured components, and every field stores the breakdown, so any number can be
explained term by term. Each field stores its breakdown, so any score can be explained term by term.

| Component | Weight | Source |
|---|---|---|
| `visual_clarity` | 0.20 | Stage 1 page metrics — bounds every field on that page |
| `evidence_support` | 0.30 | character agreement between `value` and `evidence`, minus a `?` penalty |
| `token_confidence` | 0.20 | mean `exp(logprob)` of the tokens that generated the value |
| `format_validity` | 0.20 | Stage 4 validator verdict |
| `consistency` | 0.10 | cross-field, MRZ agreement, agreement across attempts and pages |

`token_confidence` is the piece worth highlighting: it is the model's *actual* distribution over
the characters it emitted, recovered from `output_scores` and aligned back to the value's
character span. Unlike a self-reported label, it cannot be produced fluently — a model that was
genuinely torn between `0` and `O` shows it here.

Gates can only lower a score: negative status → `UNREADABLE`; `UNCERTAIN` → cap 0.55; `?` in
evidence → cap 0.45; invalid format → cap 0.50; missing evidence → cap 0.40; handwriting → cap 0.60.

Bands: HIGH ≥ 0.80, MEDIUM ≥ 0.60, LOW ≥ 0.40, else UNREADABLE. **These are defaults, not
findings** — Stage 8's `calibrate_thresholds()` fits them to your labelled sample.

**Integration.** The band drives the retry decision (Stage 6), the merge across pages and the
document verdict (Stage 7).

In [ ]:
# =========================================================================
# STAGE 5.1 — TOKEN-LEVEL CONFIDENCE FROM output_scores
# =========================================================================
def token_confidence_for_value(gen: GenOutput, field_name: str, value: str) -> Optional[float]:
    """Mean probability of the tokens that actually produced this field's value.

    Aligns the value's character span in the decoded text back to generated tokens by walking
    cumulative token lengths. Returns None when the span cannot be located (then the component is
    dropped and the remaining weights are renormalised -- never silently treated as 1.0)."""
    if not gen.token_texts or not value:
        return None
    text, needle = gen.text, str(value)
    anchor = text.find(f'"{field_name}"')
    idx = text.find(needle, anchor if anchor >= 0 else 0)
    if idx < 0:
        idx = text.find(needle)
    if idx < 0:
        return None
    start_char, end_char = idx, idx + len(needle)
    # token_texts covers the generated part; gen.text starts with the prefilled "{"
    offset = len(gen.text) - sum(len(t) for t in gen.token_texts)
    pos, probs = offset, []
    for tok, lp in zip(gen.token_texts, gen.token_logprobs):
        nxt = pos + len(tok)
        if nxt > start_char and pos < end_char:
            probs.append(math.exp(lp))
        pos = nxt
        if pos >= end_char:
            break
    return round(float(np.mean(probs)), 4) if probs else None

In [ ]:
# =========================================================================
# STAGE 5.2 — FIELD CONFIDENCE SCORING
# =========================================================================
def evidence_support_score(value: Optional[str], evidence: Optional[str]) -> Optional[float]:
    if value is None:
        return None
    if not evidence:
        return 0.0
    ratio = difflib.SequenceMatcher(None, compare_key(value), compare_key(evidence)).ratio()
    penalty = 0.25 * str(evidence).count("?")
    return round(float(np.clip(ratio - penalty, 0.0, 1.0)), 3)


def format_validity_score(validation: Dict[str, Any]) -> Optional[float]:
    valid = validation.get("valid")
    if valid is None:
        return None                                   # not checkable != invalid
    base = 1.0 if valid else 0.0
    hard = [f for f in validation.get("flags", []) if f.startswith(("homoglyph_risk",
                                                                   "mrz_checksum_mismatch",
                                                                   "mixed_script_name"))]
    return round(max(0.0, base - 0.25 * len(hard)), 3)


def consistency_score(field_name: str, cross: Dict[str, Any],
                      agreement: Optional[float]) -> Optional[float]:
    parts = []
    rel = [k for k, v in cross.get("checks", {}).items() if field_name.split("_")[0] in k or
           field_name in k]
    if rel:
        parts.append(float(np.mean([1.0 if cross["checks"][k] else 0.0 for k in rel])))
    bad = [f for f in cross.get("flags", []) if field_name in f]
    if bad:
        parts.append(0.0)
    if agreement is not None:
        parts.append(agreement)
    return round(float(np.mean(parts)), 3) if parts else None


def score_field(name: str, node: Dict[str, Any], validation: Dict[str, Any],
                cross: Dict[str, Any], quality: Dict[str, Any],
                gen: Optional[GenOutput] = None, agreement: Optional[float] = None,
                cfg: Config = CFG) -> Dict[str, Any]:
    """Compute a defensible confidence. Every term is stored so the number can be explained."""
    status, value, evidence = node.get("status"), node.get("value"), node.get("evidence")

    # ---- hard gate: a negative status is terminal, whatever else looks good ----
    if status not in POSITIVE_STATUSES or value is None:
        return {"score": 0.0, "band": "UNREADABLE", "components": {},
                "caps_applied": ["negative_status"], "status": status}

    comps: Dict[str, Optional[float]] = {
        "visual_clarity":   quality.get("visual_clarity"),
        "evidence_support": evidence_support_score(value, evidence),
        "token_confidence": token_confidence_for_value(gen, name, value) if gen else None,
        "format_validity":  format_validity_score(validation),
        "consistency":      consistency_score(name, cross, agreement),
    }
    # Missing components are DROPPED and the weights renormalised -- never defaulted to 1.0,
    # which would silently reward the absence of a check.
    usable = {k: v for k, v in comps.items() if v is not None}
    total_w = sum(cfg.weights[k] for k in usable) or 1.0
    score = sum(cfg.weights[k] * v for k, v in usable.items()) / total_w

    caps: List[str] = []
    def cap(limit: float, why: str) -> None:
        nonlocal score
        if score > limit:
            score = limit
            caps.append(why)

    if status == "UNCERTAIN":
        cap(cfg.cap_uncertain, "status_uncertain")
    if evidence and "?" in evidence:
        cap(cfg.cap_evidence_gap, "illegible_evidence")
    if not evidence:
        cap(cfg.cap_no_evidence, "no_evidence")
    if validation.get("valid") is False:
        cap(cfg.cap_format_invalid, "format_invalid")
    if node.get("text_type") == "handwritten":
        cap(cfg.cap_handwritten, "handwritten")
    if "evidence_mismatch" in node.get("flags", []):
        cap(0.30, "evidence_mismatch")

    band = ("HIGH" if score >= cfg.band_high else "MEDIUM" if score >= cfg.band_medium
            else "LOW" if score >= cfg.band_low else "UNREADABLE")
    return {"score": round(float(score), 3), "band": band,
            "components": {k: v for k, v in comps.items()},
            "weights_used": {k: cfg.weights[k] for k in usable},
            "caps_applied": caps, "status": status}


def score_page(fields: Dict[str, Any], validation: Dict[str, Any], quality: Dict[str, Any],
               gen: Optional[GenOutput] = None, agreements: Dict[str, float] = None,
               cfg: Config = CFG) -> Dict[str, Any]:
    agreements = agreements or {}
    out = {}
    for name, node in fields.items():
        conf = score_field(name, node, validation["fields"].get(name, {}),
                           validation["cross_field"], quality, gen,
                           agreements.get(name), cfg)
        out[name] = {**node, "validation": validation["fields"].get(name, {}), "confidence": conf}
    return out

## Stage 6 — Targeted retry ladder

**Problem.** v1's `RETRY_ON_RAW` re-ran the *entire page* at full cost whenever the result looked
empty — doubling the cost of exactly the pages that were already the slowest, and asking for all
twelve fields again when eleven had been read perfectly.

**Solution.** Retry is scoped three ways:

* **Only failing pages.** A page whose critical fields all reach the accept band is finished.
* **Only missing fields.** The retry prompt lists just the unresolved fields, so both the prompt
  and the output shrink — `max_new_tokens` drops from 384 to 160.
* **Only recoverable pages.** A blank or near-black page is marked `UNREADABLE` immediately and
  never enters the ladder (`skip_retry_if_hopeless`). Spending 40 s to confirm a page is blank is
  the worst trade in the pipeline.

The escalation, stopping as soon as the accept band is reached:

| Attempt | Variant | Rationale |
|---|---|---|
| 1 | Adaptive preprocessing at `target_max_dimension` | the normal path |
| 2 | Orientation re-derived + targeted enhancement for the specific failed metric | fixes geometry and contrast faults |
| 3 | `retry_max_dimension` (1600) + upscale, cropped to the document region | resolution faults; the crop keeps the token count sane |
| 4 | Second independent pass on the still-shaky critical fields, compared for agreement | disagreement between passes is itself a signal and feeds `consistency` |

**Integration.** `should_retry()` reads Stage 5 bands and Stage 1 quality; each attempt's result is
merged field-by-field, keeping only *improvements*, so a good read from attempt 1 is never
overwritten by a worse read from attempt 3.

In [ ]:
# =========================================================================
# STAGE 6.1 — extract_page(), should_retry(), retry_page()
# =========================================================================
def extract_page(image: Image.Image, quality: Dict[str, Any], engine,
                 fields: Optional[Sequence[str]] = None, page_number: Optional[int] = None,
                 n_pages: Optional[int] = None, attempt_hint: Optional[str] = None,
                 trace: Optional[Trace] = None, cfg: Config = CFG) -> Dict[str, Any]:
    """One model call -> parsed, validated, scored fields."""
    fields = list(fields or ALL_FIELDS)
    prompt = build_extraction_prompt(fields, page_number, n_pages, attempt_hint)
    mnt = cfg.max_new_tokens if len(fields) > 6 else cfg.max_new_tokens_targeted

    t0 = time.perf_counter()
    try:
        gen = engine.generate(image, EXTRACTION_SYSTEM, prompt, max_new_tokens=mnt)
    except Exception as exc:                       # tokenizer edge cases, driver faults
        gen = GenOutput(text="", n_output_tokens=0, latency_s=0.0,
                        error=f"{type(exc).__name__}: {exc}",
                        error_kind="oom" if is_oom_error(exc) else "inference")
        release_cuda_cache()
    err, err_kind = gen.error, gen.error_kind
    infer_s = time.perf_counter() - t0
    if trace:
        trace.count("model_calls")
        trace.count("output_tokens", gen.n_output_tokens)
        trace.timings["model_inference"] += infer_s

    if err:
        # A failed CALL is not an unreadable DOCUMENT. The fields stay null, but they are tagged
        # as a system error so the document is routed to ERROR rather than claiming the scan was
        # illegible -- that distinction is the whole point for a KYC record.
        parsed, ok, warn = blank_page_fields(fields, f"system_error:{err_kind}"), False, err
    else:
        parsed, ok, warn = parse_page_output(gen.text, fields)

    validation = validate_extraction(parsed)
    scored = score_page(parsed, validation, quality, gen if not err else None, cfg=cfg)

    # An emergency OOM downscale means the read came from fewer pixels than intended: cap it.
    if gen.degraded:
        for node in scored.values():
            node.setdefault("flags", []).append(gen.degraded)
            conf = node.get("confidence", {})
            if conf.get("score", 0) > cfg.band_medium:
                conf["score"] = cfg.band_medium
                conf["band"] = "MEDIUM"
                conf.setdefault("caps_applied", []).append("oom_downscaled")

    return {"fields": scored, "parse_ok": ok, "warning": warn,
            "n_output_tokens": gen.n_output_tokens, "truncated": gen.truncated,
            "inference_s": round(infer_s, 2), "requested_fields": fields,
            "raw_output": gen.text[:4000], "error": err, "error_kind": err_kind,
            "degraded": gen.degraded}


BAND_RANK = {"HIGH": 3, "MEDIUM": 2, "LOW": 1, "UNREADABLE": 0}


def unresolved_fields(result: Dict[str, Any], cfg: Config = CFG) -> List[str]:
    """Fields worth another attempt: below the accept band AND not legitimately absent."""
    need = []
    for name, node in result["fields"].items():
        band = node["confidence"]["band"]
        status = node.get("status")
        if status == "NOT_PRESENT":                # the document genuinely lacks this field
            continue
        if BAND_RANK[band] < BAND_RANK[cfg.accept_band]:
            need.append(name)
    return need


def should_retry(result: Dict[str, Any], quality: Dict[str, Any], attempt: int,
                 cfg: Config = CFG) -> Tuple[bool, str, List[str]]:
    # Retrying an OOM is guaranteed to OOM again and burns minutes doing it. The engine has
    # already tried splitting and downscaling by this point.
    if result.get("error_kind"):
        return False, f"system_error:{result['error_kind']}", []
    if attempt >= cfg.max_attempts_per_page:
        return False, "attempt_budget_exhausted", []
    if quality.get("is_blank"):
        return False, "page_is_blank", []
    missing = unresolved_fields(result, cfg)
    if not missing:
        return False, "all_fields_accepted", []
    critical_missing = [f for f in missing if f in CRITICAL_FIELDS]

    # A page with no critical field pending is not worth a 20-second retry for a nice-to-have.
    if not critical_missing and attempt >= 1:
        return False, "only_non_critical_pending", missing
    if cfg.skip_retry_if_hopeless and quality.get("visual_clarity", 1.0) < 0.12 \
            and not result["parse_ok"]:
        return False, "image_hopeless", missing
    return True, "critical_fields_unresolved", missing


def _merge_attempt(base: Dict[str, Any], new: Dict[str, Any], cfg: Config = CFG) -> Dict[str, Any]:
    """Keep the better read per field. An attempt can only improve the page, never degrade it."""
    merged = dict(base["fields"])
    improved = []
    for name, node in new["fields"].items():
        old = merged.get(name)
        if old is None or node["confidence"]["score"] > old["confidence"]["score"] + 1e-9:
            merged[name] = node
            improved.append(name)
    out = dict(new)
    out["fields"] = merged
    out["improved_fields"] = improved
    return out


def retry_page(raw_image: Image.Image, base_result: Dict[str, Any], quality: Dict[str, Any],
               engine, page_number: int, n_pages: int, trace: Trace,
               cfg: Config = CFG) -> Dict[str, Any]:
    """Escalating ladder. Stops as soon as the accept band is reached."""
    result = base_result
    attempt = 1
    history = [{"attempt": 1, "variant": "adaptive", "improved": None,
                "inference_s": base_result["inference_s"]}]

    while True:
        go, reason, missing = should_retry(result, quality, attempt, cfg)
        if not go:
            result["retry_stop_reason"] = reason
            break
        attempt += 1
        fields = missing if cfg.retry_only_missing_fields else ALL_FIELDS

        with trace.timed("preprocessing"):
            if attempt == 2:
                # variant B: re-derive geometry and push contrast/denoise regardless of flags
                orient = detect_orientation(raw_image, getattr(engine, "orientation_probe", None), cfg)
                skew = estimate_skew(raw_image, cfg)
                plan = plan_preprocessing(quality, orient, skew, cfg)
                for op in ("clahe", "flatten_illumination", "denoise"):
                    if op not in plan:
                        plan.append(op)
                img, meta = preprocess_page(raw_image, quality, plan,
                                            cfg.target_max_dimension, True, cfg)
                hint = "The previous read was unreliable. Report UNREADABLE rather than guessing."
            else:
                # variant C: more pixels, cropped tighter
                orient = detect_orientation(raw_image, None, cfg)
                plan = plan_preprocessing(quality, orient, estimate_skew(raw_image, cfg), cfg)
                if "upscale" not in plan:
                    plan.append("upscale")
                img, meta = preprocess_page(raw_image, quality, plan,
                                            cfg.retry_max_dimension, True, cfg)
                hint = "Higher resolution view. Read only what is visible."

        trace.count("retries")
        new = extract_page(img, quality, engine, fields, page_number, n_pages, hint, trace, cfg)
        result = _merge_attempt(result, new)
        history.append({"attempt": attempt, "variant": "B" if attempt == 2 else "C",
                        "preprocess": meta.get("applied"),
                        "improved": result.get("improved_fields"),
                        "inference_s": new["inference_s"]})

    # ---- optional independent verification pass on the fields that are still shaky ----
    if cfg.enable_second_pass:
        shaky = [f for f, n in result["fields"].items()
                 if n["confidence"]["band"] == "MEDIUM" and f in CRITICAL_FIELDS]
        if shaky:
            with trace.timed("preprocessing"):
                img2, _ = preprocess_page(raw_image, quality, ["clahe"],
                                          cfg.retry_max_dimension, False, cfg)
            trace.count("second_pass")
            second = extract_page(img2, quality, engine, shaky, page_number, n_pages,
                                  "Independent verification pass.", trace, cfg)
            for f in shaky:
                a, b = result["fields"][f], second["fields"].get(f, {})
                agree = (a.get("value") is not None and
                         compare_key(a.get("value")) == compare_key(b.get("value") or ""))
                a.setdefault("flags", []).append("second_pass_agreed" if agree
                                                 else "second_pass_disagreed")
                # agreement/disagreement enters the score through the consistency component
                a["confidence"] = score_field(f, a, a.get("validation", {}),
                                              {"checks": {}, "flags": []}, quality,
                                              agreement=1.0 if agree else 0.0, cfg=cfg)
                if not agree:
                    a["confidence"]["caps_applied"].append("second_pass_disagreed")
                    a["confidence"]["score"] = min(a["confidence"]["score"], 0.35)
                    a["confidence"]["band"] = "LOW"
            history.append({"attempt": attempt + 1, "variant": "verification",
                            "fields": shaky, "inference_s": second["inference_s"]})

    result["attempts"] = history
    result["n_attempts"] = len(history)
    return result

## Stage 7 — Page-level processing and document aggregation

**Problem.** v1 treated the document as the unit of work: one bad page contaminated the whole run,
and there was no way to reprocess a single page.

**Solution.** Each page is an independent unit with its own quality record, preprocessing plan,
attempts and results, saved as it completes. Aggregation then picks, per field, the highest-scoring
read across pages — a passport's data page and its MRZ page contribute different fields, which is
the normal case, not an exception.

Conflict handling is unchanged in spirit from v1 but now graded: two reads that disagree at
comparable confidence yield `null` plus `MANUAL_REVIEW`, because there is no evidence to prefer
one. A clearly stronger read wins and records the loser.

**Document verdict:**

| Verdict | Rule |
|---|---|
| `ERROR` | a page failed for infrastructure reasons (CUDA OOM, driver fault). Checked **first**, so a system failure is never recorded as a judgement about the scan |
| `MANUAL_REVIEW` | any critical field unreadable, any cross-page conflict, any evidence mismatch, or a failed MRZ checksum |
| `UNREADABLE` | no critical field reached the accept band anywhere in the document |
| `LOW_CONFIDENCE` | all critical fields present but at least one sits below `HIGH` |
| `PASS` | every critical field at `HIGH`, no validation flags |

**Batching.** Pages of the same document are grouped into `page_batch_size` batches for the first
attempt, which is where the H100 pays off: decoding at batch 1 wastes most of the memory bandwidth.
Batching falls back to sequential automatically on OOM.

In [ ]:
# =========================================================================
# STAGE 7.1 — PDF LOADING AND PAGE RENDERING
# =========================================================================
def load_pdf(pdf_path: Path):
    if fitz is None:
        raise RuntimeError("PyMuPDF is required to render PDFs")
    return fitz.open(str(pdf_path))


def render_pdf_pages(pdf_path: Path, cfg: Config = CFG) -> List[Image.Image]:
    doc = load_pdf(pdf_path)
    pages = []
    try:
        for i, page in enumerate(doc):
            if i >= cfg.max_pages:
                break
            pix = page.get_pixmap(dpi=cfg.render_dpi, colorspace=fitz.csRGB, alpha=False)
            pages.append(Image.frombytes("RGB", (pix.width, pix.height), pix.samples).copy())
    finally:
        doc.close()
    return pages


def prepare_page(img: Image.Image, engine, cfg: Config = CFG) -> Tuple[Image.Image, Dict, Dict]:
    """Quality -> orientation -> plan -> preprocess. All CPU, no model call in the common case."""
    quality = analyze_page_quality(img, cfg.render_dpi, cfg)
    if quality["is_blank"]:
        return img, quality, {"applied": [], "skipped": "blank_page"}
    orient = detect_orientation(img, getattr(engine, "orientation_probe", None), cfg)
    skew = estimate_skew(img, cfg)
    quality["orientation"] = orient
    quality["skew_deg"] = skew
    quality["needs_deskew"] = bool(abs(skew) >= cfg.deskew_min_deg)
    plan = plan_preprocessing(quality, orient, skew, cfg)
    processed, meta = preprocess_page(img, quality, plan, cfg.target_max_dimension, True, cfg)
    meta["plan"] = plan
    meta["orientation"] = orient
    return processed, quality, meta

In [ ]:
# =========================================================================
# STAGE 7.2 — DOCUMENT PROCESSING AND AGGREGATION
# =========================================================================
def aggregate_document_results(page_records: List[Dict[str, Any]],
                               cfg: Config = CFG) -> Tuple[Dict[str, Any], List[Dict[str, Any]]]:
    """Combine page-level fields into one document-level answer, keeping full provenance."""
    final, conflicts = {}, []
    for name in ALL_FIELDS:
        cands = []
        for rec in page_records:
            node = rec["result"]["fields"].get(name)
            if not node or node.get("value") is None:
                continue
            if BAND_RANK[node["confidence"]["band"]] < BAND_RANK[cfg.accept_band]:
                continue
            cands.append({"page": rec["page_number"], "node": node,
                          "score": node["confidence"]["score"]})
        if not cands:
            # Three different reasons for an empty field, and they must not be conflated:
            # the document has no such field, the scan was illegible, or our GPU failed.
            statuses = [rec["result"]["fields"].get(name, {}).get("status")
                        for rec in page_records]
            errored = [rec for rec in page_records if rec["result"].get("error_kind")]
            if errored and len(errored) == len(page_records):
                status = "SYSTEM_ERROR"
            elif statuses and all(s == "NOT_PRESENT" for s in statuses):
                status = "NOT_PRESENT"
            else:
                status = "UNREADABLE"
            final[name] = {"value": None, "status": status, "confidence_band": "UNREADABLE",
                           "confidence_score": 0.0, "source_pages": [], "evidence": None}
            continue

        cands.sort(key=lambda c: -c["score"])
        best = cands[0]
        rivals = [c for c in cands[1:]
                  if compare_key(c["node"]["value"]) != compare_key(best["node"]["value"])]
        close = [c for c in rivals if best["score"] - c["score"] < 0.15]

        # A bilingual identity document legitimately carries the same field twice in two
        # scripts (PARIS / <arabic>). Those are not contradictory readings, so they are kept
        # side by side instead of flooding the review queue. Only same-script disagreement is
        # a real conflict.
        alternates = [c for c in close
                      if (c["node"].get("script") or "?") != (best["node"].get("script") or "?")]
        close = [c for c in close if c not in alternates]
        if close:
            conflicts.append({"field": name,
                              "candidates": [{"page": c["page"], "value": c["node"]["value"],
                                              "score": c["score"]} for c in cands]})
            final[name] = {"value": None, "status": "CONFLICT", "confidence_band": "UNREADABLE",
                           "confidence_score": 0.0, "source_pages": [c["page"] for c in cands],
                           "evidence": None, "needs_review": True}
            continue

        node = best["node"]
        final[name] = {
            "value": node["value"], "status": node["status"],
            "confidence_band": node["confidence"]["band"],
            "confidence_score": node["confidence"]["score"],
            "confidence_components": node["confidence"]["components"],
            "caps_applied": node["confidence"]["caps_applied"],
            "evidence": node.get("evidence"), "text_type": node.get("text_type"),
            "script": node.get("script"), "source_pages": [best["page"]],
            "alternate_script_values": [{"page": c["page"], "value": c["node"]["value"],
                                         "script": c["node"].get("script"),
                                         "score": c["score"]} for c in alternates],
            "validation": node.get("validation", {}), "flags": node.get("flags", []),
            "superseded": [{"page": c["page"], "value": c["node"]["value"], "score": c["score"]}
                           for c in rivals if c not in alternates],
        }
    return final, conflicts


def decide_document_status(final: Dict[str, Any], conflicts: List[Dict[str, Any]],
                           page_records: List[Dict[str, Any]],
                           cfg: Config = CFG) -> Dict[str, Any]:
    reasons = []
    crit = {f: final.get(f, {}) for f in cfg.critical_fields}
    unreadable_crit = [f for f, n in crit.items() if n.get("value") is None]
    low_crit = [f for f, n in crit.items() if n.get("confidence_band") in ("LOW", "MEDIUM")]
    bad_flags = [f"{f}:{fl}" for f, n in final.items() for fl in n.get("flags", [])
                 if fl in ("evidence_mismatch", "second_pass_disagreed",
                           "value_despite_illegible_evidence")]
    val_flags = [f"{f}:{fl}" for f, n in final.items()
                 for fl in (n.get("validation", {}) or {}).get("flags", [])
                 if fl.startswith(("mrz_checksum_mismatch", "implausible_age", "digits_in_name"))]

    if conflicts:
        reasons.append(f"page_conflicts:{','.join(c['field'] for c in conflicts)}")
    if len(unreadable_crit) > cfg.max_unreadable_critical:
        reasons.append("unreadable_critical:" + ",".join(unreadable_crit))
    reasons += bad_flags + val_flags

    # A system failure must never be reported as a statement about the document.
    sys_err = sorted({p["result"].get("error_kind") for p in page_records
                      if p["result"].get("error_kind")})
    err_pages = [p["page_number"] for p in page_records if p["result"].get("error_kind")]
    if sys_err:
        reasons.insert(0, f"system_error:{','.join(sys_err)}:pages={err_pages}")
        status = "ERROR"
    elif len(unreadable_crit) == len(cfg.critical_fields):
        status = "UNREADABLE"
    elif reasons:
        status = "MANUAL_REVIEW"
    elif low_crit:
        status = "LOW_CONFIDENCE"
        reasons.append("critical_below_high:" + ",".join(low_crit))
    else:
        status = "PASS"
    return {"status": status, "reasons": reasons,
            "needs_manual_review": status in ("MANUAL_REVIEW", "UNREADABLE", "ERROR")}


def process_document(customer_id: str, pdf_path: Path, engine, cfg: Config = CFG) -> Dict[str, Any]:
    """Page-level pipeline: quality -> preprocess -> extract -> retry -> aggregate."""
    trace = Trace(customer_id, cfg.doc_name)
    with trace.timed("pdf_render"):
        pages = render_pdf_pages(pdf_path, cfg)
    trace.count("pages", len(pages))

    prepared = []
    for i, raw in enumerate(pages, start=1):
        t_prep = time.perf_counter()
        with trace.timed("quality_and_preprocessing"):
            proc, quality, meta = prepare_page(raw, engine, cfg)
        meta["preprocess_time_s"] = round(time.perf_counter() - t_prep, 3)
        prepared.append({"page_number": i, "raw": raw, "image": proc,
                         "quality": quality, "preprocess": meta})

    # ---- first attempt: batch the non-blank pages ----
    todo = [p for p in prepared if not p["quality"]["is_blank"]]
    first: Dict[int, Dict[str, Any]] = {}
    bs = max(1, cfg.page_batch_size)
    for i in range(0, len(todo), bs):
        chunk = todo[i:i + bs]
        prompts = [build_extraction_prompt(ALL_FIELDS, p["page_number"], len(pages))
                   for p in chunk]
        batched = False
        if len(chunk) > 1:
            try:
                t0 = time.perf_counter()
                gens = engine.generate_batch([p["image"] for p in chunk], EXTRACTION_SYSTEM,
                                             prompts, max_new_tokens=cfg.max_new_tokens)
                trace.timings["model_inference"] += time.perf_counter() - t0
                trace.count("model_calls", len(chunk))
                trace.count("batched_calls", 1)
                if any(g.error_kind for g in gens):
                    raise RuntimeError("batch returned a system error; retrying sequentially")
                for p, gen in zip(chunk, gens):
                    trace.count("output_tokens", gen.n_output_tokens)
                    parsed, ok, warn = parse_page_output(gen.text, ALL_FIELDS)
                    validation = validate_extraction(parsed)
                    first[p["page_number"]] = {
                        "fields": score_page(parsed, validation, p["quality"], gen, cfg=cfg),
                        "parse_ok": ok, "warning": warn, "n_output_tokens": gen.n_output_tokens,
                        "truncated": gen.truncated, "inference_s": gen.latency_s,
                        "requested_fields": ALL_FIELDS, "raw_output": gen.text[:4000],
                        "error": None}
                batched = True
            except Exception as exc:
                # OOM is the expected failure here: drop to one page at a time rather than
                # losing the document. The fallback is logged so batch sizing stays tunable.
                log.warning("batch of %d failed (%s); falling back to sequential",
                            len(chunk), type(exc).__name__)
                if torch is not None and torch.cuda.is_available():
                    torch.cuda.empty_cache()
        if not batched:
            for p in chunk:
                first[p["page_number"]] = extract_page(p["image"], p["quality"], engine,
                                                       ALL_FIELDS, p["page_number"], len(pages),
                                                       None, trace, cfg)

    # ---- per-page retry ladder, only where needed ----
    page_records = []
    for p in prepared:
        t_page = time.perf_counter()
        if p["quality"]["is_blank"]:
            result = {"fields": blank_page_fields(ALL_FIELDS, "blank_page"), "parse_ok": True,
                      "warning": "blank_page", "n_output_tokens": 0, "inference_s": 0.0,
                      "attempts": [], "n_attempts": 0, "retry_stop_reason": "page_is_blank",
                      "error": None, "requested_fields": []}
            for node in result["fields"].values():
                node["confidence"] = {"score": 0.0, "band": "UNREADABLE", "components": {},
                                      "caps_applied": ["blank_page"], "status": node["status"]}
        else:
            result = first[p["page_number"]]
            result = retry_page(p["raw"], result, p["quality"], engine,
                                p["page_number"], len(pages), trace, cfg)

        needs_review = any(n["confidence"]["band"] in ("LOW", "UNREADABLE")
                           for f, n in result["fields"].items() if f in cfg.critical_fields)
        # The first pass may have run in a batch BEFORE this loop, so its inference time is not
        # inside t_page. Add it back, otherwise batched pages report a misleading ~0 s.
        first_inference = float(first.get(p["page_number"], {}).get("inference_s", 0.0) or 0.0)
        page_total = round((time.perf_counter() - t_page) + first_inference +
                           float(p["preprocess"].get("preprocess_time_s", 0.0)), 2)
        page_records.append({
            "page_number": p["page_number"], "quality": p["quality"],
            "preprocessing": p["preprocess"], "result": result,
            "needs_review": needs_review,
            "processing_time_s": page_total})
        trace.page_times[f"page_{p['page_number']}"] = page_total

    final, conflicts = aggregate_document_results(page_records, cfg)
    verdict = decide_document_status(final, conflicts, page_records, cfg)

    return {
        "customer_id": customer_id,
        "document": cfg.doc_name,
        "source_pdf": str(pdf_path),
        "source_sha256": hashlib.sha256(Path(pdf_path).read_bytes()).hexdigest(),
        "run_id": RUN_ID, "extracted_at_utc": utcnow(),
        "engine": {"name": getattr(engine, "name", "?"), "prompt_hashes": PROMPT_HASHES,
                   "max_new_tokens": cfg.max_new_tokens,
                   "repetition_penalty": cfg.repetition_penalty,
                   "target_max_dimension": cfg.target_max_dimension,
                   "page_batch_size": cfg.page_batch_size},
        "n_pages": len(pages),
        "pages": [{k: v for k, v in r.items()} for r in page_records],
        "final_fields": final,
        "conflicts": conflicts,
        "document_confidence": verdict["status"],
        "review_reasons": verdict["reasons"],
        "needs_manual_review": verdict["needs_manual_review"],
        "trace": trace.summary(),
    }


def save_results(record: Dict[str, Any], cfg: Config = CFG) -> Path:
    out = DIRS["results"] / f"{record['customer_id']}.json"
    slim = json.loads(json.dumps(record, ensure_ascii=False, default=str))
    out.write_text(json.dumps(slim, ensure_ascii=False, indent=2), encoding="utf-8")
    with (DIRS["logs"] / f"run_{RUN_ID}.jsonl").open("a", encoding="utf-8") as fh:
        fh.write(json.dumps({
            "customer_id": record["customer_id"], "document": record["document"],
            "number_of_pages": record["n_pages"],
            "processing_time_s": record["trace"]["processing_time_s"],
            "page_processing_times": record["trace"]["page_processing_times_s"],
            "preprocessing_time_s": record["trace"]["timings_s"].get("quality_and_preprocessing", 0)
                                    + record["trace"]["timings_s"].get("preprocessing", 0),
            "model_inference_time_s": record["trace"]["timings_s"].get("model_inference", 0),
            "number_of_model_calls": record["trace"]["counts"].get("model_calls", 0),
            "number_of_retries": record["trace"]["counts"].get("retries", 0),
            "output_tokens": record["trace"]["counts"].get("output_tokens", 0),
            "pages_retried": [p["page_number"] for p in record["pages"]
                              if p["result"].get("n_attempts", 0) > 1],
            "fields_with_low_confidence": [f for f, n in record["final_fields"].items()
                                           if n["confidence_band"] in ("LOW", "UNREADABLE")],
            "final_status": record["document_confidence"],
        }, ensure_ascii=False) + "\n")
    return out

In [ ]:
# =========================================================================
# STAGE 7.3 — BATCH RUNNER OVER TARGET CUSTOMERS
# =========================================================================
def find_target_documents(cfg: Config = CFG) -> List[Tuple[str, Path]]:
    """Reads the output of the v1 stages 1-3: SELECTED_DIR/<customer>/<doc_key>.pdf"""
    out = []
    if not cfg.selected_dir.exists():
        log.warning("selected_dir does not exist: %s", cfg.selected_dir)
        return out
    for d in sorted(p for p in cfg.selected_dir.iterdir() if p.is_dir()):
        pdf = d / f"{cfg.doc_key}.pdf"
        if pdf.exists():
            out.append((d.name, pdf))
    return out


def run_batch(targets=None, limit: Optional[int] = None, resume: bool = True,
              engine=None, cfg: Config = CFG) -> pd.DataFrame:
    engine = engine or ENGINE
    targets = targets if targets is not None else find_target_documents(cfg)
    if limit:
        targets = targets[:limit]
    rows, consecutive_errors = [], 0
    for i, (cid, pdf) in enumerate(targets, 1):
        dest = DIRS["results"] / f"{cid}.json"
        if resume and dest.exists():
            log.info("[%d/%d] %s skipped (done)", i, len(targets), cid)
            continue
        try:
            rec = process_document(cid, pdf, engine, cfg)
            save_results(rec, cfg)
            t = rec["trace"]
            log.info("[%d/%d] %s  %s  %dp  %.1fs  calls=%d retries=%d tokens=%d",
                     i, len(targets), cid, rec["document_confidence"], rec["n_pages"],
                     t["processing_time_s"], t["counts"].get("model_calls", 0),
                     t["counts"].get("retries", 0), t["counts"].get("output_tokens", 0))
            rows.append({"customer_id": cid, "status": rec["document_confidence"],
                         "pages": rec["n_pages"], "time_s": t["processing_time_s"],
                         "model_calls": t["counts"].get("model_calls", 0),
                         "retries": t["counts"].get("retries", 0),
                         "output_tokens": t["counts"].get("output_tokens", 0)})

            # Circuit breaker: if the GPU is out of memory, every remaining document will fail
            # the same way. Stop and say so, instead of writing hundreds of ERROR records.
            if rec["document_confidence"] == "ERROR":
                consecutive_errors += 1
                if consecutive_errors >= cfg.max_consecutive_error_docs:
                    log.error("ABORTING BATCH: %d consecutive documents failed with system "
                              "errors (usually CUDA OOM). Nothing is wrong with the documents. "
                              "Run gpu_report(), then free_model() or restart the kernel, raise "
                              "CFG.reserve_vram_gib, or lower CFG.max_visual_tokens.",
                              consecutive_errors)
                    break
            else:
                consecutive_errors = 0
        except Exception as exc:
            log.error("[%d/%d] %s FAILED: %s", i, len(targets), cid, exc)
            (DIRS["logs"] / f"{cid}.error.txt").write_text(traceback.format_exc(), encoding="utf-8")
            rows.append({"customer_id": cid, "status": "ERROR", "pages": 0, "time_s": 0.0,
                         "model_calls": 0, "retries": 0, "output_tokens": 0})
    return pd.DataFrame(rows)


# TARGETS = find_target_documents(CFG)
# batch_df = run_batch(TARGETS, limit=5)
# display(batch_df)
print("ready: call run_batch(find_target_documents(CFG), limit=5) to process documents")

## Stage 8 — Logging, statistics, benchmarking, calibration

**Problem.** "Why does a document take 6 minutes?" and "is v2 actually better?" were both
unanswerable. Any claim of improvement without measurement is just a story.

**Solution.** Three tools:

* `dataset_statistics()` — P50/P95 latency, model calls per document, retry rate, unreadable rate,
  review rate, low-confidence field rate, plus the **time breakdown by stage** that answers the six
  minutes question directly.
* `run_benchmark()` — runs the same sample under several configurations and reports accuracy,
  latency and, most importantly, **silent error rate**: wrong values accepted at HIGH or MEDIUM
  confidence. For KYC that is the number that matters. A refused field costs an operator two
  minutes; a confidently wrong one enters the customer record.
* `calibrate_thresholds()` and `sweep_resolution()` — fit the confidence bands and
  `target_max_dimension` to your own labelled sample instead of inheriting my defaults.

Ground truth format — one row per customer, one column per field:

```csv
customer_id,surname,given_names,date_of_birth,document_number
CUST0001,DUPONT,JEAN PIERRE,01.01.1980,12AB45678
CUST0002,,,,                      # leave blank where the truth is genuinely unreadable
```

Comparison uses `compare_key()` (case, accent and punctuation insensitive) so `Jean-Pierre` and
`JEAN PIERRE` match. That normalisation exists **only** in the comparator — the pipeline never
normalises a stored value.

In [ ]:
# =========================================================================
# STAGE 8.1 — DATASET STATISTICS (answers "where do the minutes go?")
# =========================================================================
def load_results(results_dir: Path = None) -> List[Dict[str, Any]]:
    results_dir = results_dir or DIRS["results"]
    out = []
    for p in sorted(Path(results_dir).glob("*.json")):
        try:
            out.append(json.loads(p.read_text(encoding="utf-8")))
        except Exception as exc:
            log.warning("unreadable result %s: %s", p.name, exc)
    return out


def dataset_statistics(records: List[Dict[str, Any]] = None, cfg: Config = CFG) -> Dict[str, Any]:
    records = records if records is not None else load_results()
    if not records:
        return {"n_documents": 0}
    times = np.array([r["trace"]["processing_time_s"] for r in records], dtype=float)
    calls = np.array([r["trace"]["counts"].get("model_calls", 0) for r in records], dtype=float)
    toks = np.array([r["trace"]["counts"].get("output_tokens", 0) for r in records], dtype=float)
    pages = np.array([r["n_pages"] for r in records], dtype=float)

    all_pages = [p for r in records for p in r["pages"]]
    retried = [p for p in all_pages if p["result"].get("n_attempts", 0) > 1]
    blank = [p for p in all_pages if p["quality"].get("is_blank")]
    # Pages that failed for infrastructure reasons are counted as system errors, NOT as
    # unreadable: mixing them would corrupt the corpus quality metric with GPU problems.
    unreadable_pages = [p for p in all_pages
                        if not p["result"].get("error_kind")
                        and all(n["confidence"]["band"] == "UNREADABLE"
                                for f, n in p["result"]["fields"].items()
                                if f in cfg.critical_fields)]
    field_bands = [n["confidence_band"] for r in records for n in r["final_fields"].values()]

    stage = defaultdict(float)
    for r in records:
        for k, v in r["trace"]["timings_s"].items():
            stage[k] += v
    total_stage = sum(stage.values()) or 1.0

    stats = {
        "n_documents": len(records),
        "n_pages": int(pages.sum()),
        "avg_processing_time_s": round(float(times.mean()), 1),
        "median_processing_time_s": round(float(np.median(times)), 1),
        "p95_processing_time_s": round(float(np.percentile(times, 95)), 1),
        "avg_time_per_page_s": round(float(times.sum() / max(1, pages.sum())), 1),
        "avg_model_calls_per_doc": round(float(calls.mean()), 2),
        "avg_output_tokens_per_call": round(float(toks.sum() / max(1, calls.sum())), 1),
        "retry_rate_pages": round(len(retried) / max(1, len(all_pages)), 3),
        "blank_page_rate": round(len(blank) / max(1, len(all_pages)), 3),
        "unreadable_page_rate": round(len(unreadable_pages) / max(1, len(all_pages)), 3),
        "manual_review_rate": round(np.mean([r["needs_manual_review"] for r in records]), 3),
        "system_error_rate": round(float(np.mean([r["document_confidence"] == "ERROR"
                                                  for r in records])), 3),
        "oom_page_rate": round(len([p for p in all_pages
                                    if p["result"].get("error_kind") == "oom"]) /
                               max(1, len(all_pages)), 3),
        "status_distribution": dict(pd.Series([r["document_confidence"] for r in records])
                                    .value_counts()),
        "low_confidence_field_rate": round(
            float(np.mean([b in ("LOW", "UNREADABLE") for b in field_bands])), 3),
        "time_breakdown_pct": {k: round(100 * v / total_stage, 1)
                               for k, v in sorted(stage.items(), key=lambda x: -x[1])},
    }
    return stats


def print_statistics(stats: Dict[str, Any]) -> None:
    if not stats.get("n_documents"):
        print("no results yet")
        return
    print("=" * 62)
    for k in ["n_documents", "n_pages", "avg_processing_time_s", "median_processing_time_s",
              "p95_processing_time_s", "avg_time_per_page_s", "avg_model_calls_per_doc",
              "avg_output_tokens_per_call", "retry_rate_pages", "unreadable_page_rate",
              "manual_review_rate", "system_error_rate", "oom_page_rate",
              "low_confidence_field_rate"]:
        print(f"{k:<32} {stats[k]}")
    print("-" * 62)
    print("time breakdown (% of measured stage time) -- this answers 'where do the minutes go':")
    for k, v in stats["time_breakdown_pct"].items():
        print(f"  {k:<30} {v:>5} %")
    print("status distribution:", stats["status_distribution"])
    print("=" * 62)

In [ ]:
# =========================================================================
# STAGE 8.2 — BENCHMARK: accuracy, silent errors, latency
# =========================================================================
def load_ground_truth(csv_path: Path) -> pd.DataFrame:
    gt = pd.read_csv(csv_path, dtype=str).fillna("")
    gt["customer_id"] = gt["customer_id"].astype(str)
    return gt.set_index("customer_id")


def evaluate_against_ground_truth(records: List[Dict[str, Any]], gt: pd.DataFrame,
                                  cfg: Config = CFG) -> Dict[str, Any]:
    """Field-level accuracy plus the metric that matters for KYC: silent error rate."""
    rows = []
    for r in records:
        cid = str(r["customer_id"])
        if cid not in gt.index:
            continue
        for fname in gt.columns:
            if fname not in r["final_fields"]:
                continue
            truth = str(gt.loc[cid, fname] or "").strip()
            node = r["final_fields"][fname]
            pred, band = node.get("value"), node["confidence_band"]
            accepted = BAND_RANK[band] >= BAND_RANK[cfg.accept_band]
            correct = (compare_key(pred or "") == compare_key(truth)) if truth else (pred is None)
            rows.append({"customer_id": cid, "field": fname, "truth": truth, "pred": pred,
                         "band": band, "accepted": accepted, "correct": correct,
                         "truth_available": bool(truth)})
    df = pd.DataFrame(rows)
    if df.empty:
        return {"n_fields": 0}
    acc = df[df["accepted"]]
    known = df[df["truth_available"]]
    return {
        "n_fields": len(df),
        "coverage": round(float(df["accepted"].mean()), 3),
        "accuracy_all": round(float(known["correct"].mean()), 3) if len(known) else None,
        "accuracy_accepted": round(float(acc["correct"].mean()), 3) if len(acc) else None,
        # THE KYC METRIC: wrong data that the system was confident about
        "silent_error_rate": round(float((~acc["correct"]).mean()), 4) if len(acc) else None,
        "refusal_rate": round(float((~df["accepted"]).mean()), 3),
        "per_field": df.groupby("field").agg(
            n=("correct", "size"), coverage=("accepted", "mean"),
            accuracy_accepted=("correct", lambda s: s[df.loc[s.index, "accepted"]].mean()
                               if df.loc[s.index, "accepted"].any() else np.nan)).round(3),
        "detail": df,
    }


def run_benchmark(targets, variants: Dict[str, Dict[str, Any]], gt_path: Optional[Path] = None,
                  engine=None, base_cfg: Config = CFG) -> pd.DataFrame:
    """Run the same sample under several configurations and compare. No claims without numbers.

    variants: {"baseline": {"max_new_tokens": 1024, "page_batch_size": 1, ...}, "v2": {}}
    """
    engine = engine or ENGINE
    gt = load_ground_truth(gt_path) if gt_path else None
    summary = []
    for vname, overrides in variants.items():
        cfg = Config(**{**asdict(base_cfg), **overrides})
        recs, t0 = [], time.perf_counter()
        for cid, pdf in targets:
            try:
                recs.append(process_document(cid, pdf, engine, cfg))
            except Exception as exc:
                log.error("benchmark %s / %s failed: %s", vname, cid, exc)
        wall = time.perf_counter() - t0
        (DIRS["bench"] / f"{vname}_{RUN_ID}.json").write_text(
            json.dumps(recs, ensure_ascii=False, indent=2, default=str), encoding="utf-8")

        stats = dataset_statistics(recs, cfg)
        row = {"variant": vname, "n_docs": len(recs), "wall_s": round(wall, 1),
               "avg_time_s": stats.get("avg_processing_time_s"),
               "p95_time_s": stats.get("p95_processing_time_s"),
               "calls_per_doc": stats.get("avg_model_calls_per_doc"),
               "tokens_per_call": stats.get("avg_output_tokens_per_call"),
               "retry_rate": stats.get("retry_rate_pages"),
               "unreadable_page_rate": stats.get("unreadable_page_rate"),
               "manual_review_rate": stats.get("manual_review_rate")}
        if gt is not None:
            ev = evaluate_against_ground_truth(recs, gt, cfg)
            row.update({"accuracy_accepted": ev.get("accuracy_accepted"),
                        "coverage": ev.get("coverage"),
                        "silent_error_rate": ev.get("silent_error_rate")})
        summary.append(row)
    df = pd.DataFrame(summary)
    df.to_csv(DIRS["bench"] / f"benchmark_{RUN_ID}.csv", index=False, encoding="utf-8-sig")
    return df


BENCHMARK_VARIANTS = {
    # reproduces the v1 behaviour for an apples-to-apples comparison
    "baseline_v1_like": {"max_new_tokens": 1024, "page_batch_size": 1,
                         "max_attempts_per_page": 1, "enable_second_pass": False,
                         "target_max_dimension": 1600, "max_visual_tokens": 2300},
    "v2_default": {},
    "v2_fast": {"target_max_dimension": 1024, "max_visual_tokens": 900, "page_batch_size": 4},
}
print("benchmark variants ready:", list(BENCHMARK_VARIANTS))
print("usage: run_benchmark(find_target_documents(CFG)[:20], BENCHMARK_VARIANTS, "
      "gt_path=Path('ground_truth.csv'))")

In [ ]:
# =========================================================================
# STAGE 8.3 — CALIBRATION: fit the thresholds to YOUR data, do not inherit mine
# =========================================================================
def calibrate_thresholds(records: List[Dict[str, Any]], gt: pd.DataFrame,
                         max_silent_error: float = 0.01,
                         grid: Sequence[float] = np.arange(0.35, 0.95, 0.05),
                         cfg: Config = CFG) -> pd.DataFrame:
    """Sweep the accept threshold and report the coverage/silent-error trade-off.

    Pick the LOWEST threshold whose silent_error_rate stays under your tolerance: that maximises
    automation without letting wrong identity data through."""
    rows = []
    pairs = []
    for r in records:
        cid = str(r["customer_id"])
        if cid not in gt.index:
            continue
        for fname in gt.columns:
            node = r["final_fields"].get(fname)
            if not node:
                continue
            truth = str(gt.loc[cid, fname] or "").strip()
            pairs.append((node.get("confidence_score", 0.0), node.get("value"), truth))
    for thr in grid:
        acc = [(p, t) for s, p, t in pairs if s >= thr and p is not None]
        if not acc:
            rows.append({"threshold": round(float(thr), 2), "coverage": 0.0,
                         "accuracy": None, "silent_error_rate": None})
            continue
        correct = [compare_key(p) == compare_key(t) for p, t in acc if t]
        rows.append({
            "threshold": round(float(thr), 2),
            "coverage": round(len(acc) / max(1, len(pairs)), 3),
            "accuracy": round(float(np.mean(correct)), 3) if correct else None,
            "silent_error_rate": round(float(1 - np.mean(correct)), 4) if correct else None,
        })
    df = pd.DataFrame(rows)
    ok = df[(df["silent_error_rate"].notna()) & (df["silent_error_rate"] <= max_silent_error)]
    if not ok.empty:
        best = ok.sort_values("threshold").iloc[0]
        print(f"recommended accept threshold: {best['threshold']} "
              f"(coverage {best['coverage']}, silent errors {best['silent_error_rate']})")
    else:
        print(f"no threshold reaches silent_error_rate <= {max_silent_error} on this sample")
    return df


def sweep_resolution(targets, dimensions: Sequence[int] = (1024, 1280, 1600, 2000),
                     gt_path: Optional[Path] = None, engine=None,
                     base_cfg: Config = CFG) -> pd.DataFrame:
    """Find the smallest TARGET_MAX_DIMENSION whose accuracy matches the largest.

    Sending bigger images is not free: visual_tokens ~= W*H/784, and prefill scales with it."""
    variants = {f"dim_{d}": {"target_max_dimension": d,
                             "max_visual_tokens": int(d * d * 0.7 / 784)} for d in dimensions}
    return run_benchmark(targets, variants, gt_path, engine, base_cfg)


print("calibration helpers ready: calibrate_thresholds(), sweep_resolution()")

## Where the six minutes go, and what to do about it on an H100

Derived in section 0.2 at the top of this notebook. Restated here next to the benchmark that
verifies it — these are **targets to measure with `run_benchmark()`**, not measurements:

| # | Change | Mechanism | Expected |
|---|---|---|---|
| 1 | Thinking off, `max_new_tokens` 384, brace stop, `{` prefill | ~60% of wall-clock was decoding; this cuts tokens per call by 2–4× | largest single win |
| 2 | CV orientation cascade | removes one full model call per page | −15–25% |
| 3 | Targeted retries (failed pages only, missing fields only, 160 tokens) | retries stop costing a full page | −10–20% on retrying docs |
| 4 | Blank-page triage | zero GPU calls for empty pages | corpus dependent |
| 5 | Preprocess after downscaling | CLAHE + bilateral on 1.2 MP not 8.7 MP | −8–12 s/doc of CPU |
| 6 | `max_pixels` cap + document crop | shorter prefill, better effective resolution | −5–10% |
| 7 | `page_batch_size=4` | batch-1 decode wastes H100 bandwidth; batching amortises weight reads | −20–40% multi-page |

**Batching and memory.** A 27B bf16 model is ≈54 GB of weights; on an 80 GB H100 that leaves
~20–25 GB for activations and KV cache. At ~1 500 visual tokens per page, batch 4 is comfortable
and batch 8 is the point to start watching `nvidia-smi`. The code falls back to sequential on OOM
automatically. Verify that batching does not shift results — greedy decoding with left padding
should be stable, but run `run_benchmark()` with `page_batch_size` 1 vs 4 and confirm rather than
assume.

**If you need more than this**, the next step is a vLLM backend over the same local weights: paged
KV, continuous batching, CUDA graphs, and automatic prefix caching of the fixed system prompt
(which `transformers` re-prefills on every single call). That is typically another 2–4×, at the
cost of a second serving stack in the Domino environment. The `QwenVLEngine` interface is small —
`generate_batch()` and `orientation_probe()` — so an adapter is a contained change.

**Practical Domino notes.** Load the model once per kernel: `QwenVLEngine.get()` caches it, so
never call `load_model()` inside a loop. For large batches, run several workers over disjoint
customer slices — `run_batch(resume=True)` makes re-runs safe. Keep `HF_HUB_OFFLINE=1` set; the
first `from_pretrained` with `local_files_only=True` fails loudly rather than hanging on a network
timeout.

## What changed, and what was deliberately left alone

Preserved from v1 without modification: the ZIP extraction, filename matching, presence/absence
report, target-customer selection, the local Qwen checkpoint, the offline constraint, Domino
compatibility, and the fail-closed philosophy.

| # | CURRENT (v1) | PROBLEM | MODIFICATION | EXPECTED BENEFIT |
|---|---|---|---|---|
| 1 | `max_new_tokens=1024`, no stop, thinking on | 60% of runtime decoding unread tokens | 384 + brace stop + `enable_thinking=False` + `{` prefill | biggest latency cut; less room to talk itself into a guess |
| 2 | repetition penalty unreviewed | >1.0 corrupts `1980`, `<<<`, repeated digits | pinned to 1.0, `no_repeat_ngram_size=0` | removes a silent corruption path |
| 3 | VLM call for orientation | full prefill for 4 tokens, every page | CV cascade; VLM only on ties | −1 call/page, fallback rate measured |
| 4 | enhance at 8.7 MP then downscale | filtering discarded pixels | crop → resize → enhance | ~10 s/doc of CPU back |
| 5 | same enhancement everywhere | CLAHE damages good scans | `analyze_page_quality` → `plan_preprocessing` | fewer damaged pages, less CPU |
| 6 | `RETRY_ON_RAW` full-page rerun | doubles cost of the slowest pages | ladder, failed pages only, missing fields only | bounded, cheap retries |
| 7 | model self-reports confidence | ungrounded | 5 measured components incl. token logprobs | calibratable confidence |
| 8 | value only | nothing to check against | mandatory `evidence` + mismatch detection | direct hallucination detector |
| 9 | 4 confidence words | cannot separate absent from illegible | 5 statuses incl. `NOT_PRESENT` | accurate review queues |
| 10 | merge by confidence | no document verdict | `PASS / LOW_CONFIDENCE / UNREADABLE / MANUAL_REVIEW` | operational triage |
| 11 | aggregate timing only | six minutes unexplained | per-stage timers, call counters, JSONL | every regression attributable |
| 12 | no ground truth | improvement unprovable | benchmark + silent error rate | claims become measurements |

One honest caveat: the 180°-rotation heuristic (Stage 2, tier 3) is the least reliable component
here. It is a heuristic about where ink sits relative to a baseline, and it will be weaker on
sparse ID cards with few text lines than on dense forms. It is instrumented — `method` and `margin`
are recorded per page — so measure its fallback rate on your corpus before trusting it, and if the
VLM tie-break fires on more than ~15% of pages, install tesseract and let tier 1 do the work.